# Cell 1: Environment Setup

In [1]:
import os
import math
import copy
import time
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, balanced_accuracy_score, roc_auc_score, confusion_matrix
)


# ==============================================================================
# v15 — MULTI-SEED ENSEMBLE. The v13 configuration (v14's ordinal loss was
# rejected and is NOT carried forward) is trained with 5 independent seeds
# {42, 123, 2024, 7, 99}. Each seed goes through exactly the v13 seeding
# mechanism (set_all_seeds at condition start and right before model
# construction). This measures the real seed-to-seed variance for the first
# time, and tests whether averaging the seeds' test-set probabilities (2-5
# seed ensembles) beats any single seed. seed=42 must reproduce the known
# baseline first. The mission split keeps its own fixed random.seed(42) (Cell 4),
# so every seed sees the SAME train / val / test windows.
# ==============================================================================
SEED = 42         # the reference seed (known baseline); SEEDS below is the full list
LACA_BETA = 1.0   # LACA temporal_bias_scale — FIXED (v9 sweep winner)
PIPELINE_CONFIG = {
    "Audio bin selection": "strongest-64 (N=64 frequency bins, v6 winner; TRAINING audio only, frozen for val/test)",
    "Audio time positions": "Ta = 64 per window (FIXED, v7)",
    "Sensor input":        "raw 6 IMU channels (accel x/y/z, gyro x/y/z), Ts = 128 (FIXED; no derived magnitudes)",
    "LACA beta":           f"temporal_bias_scale = {LACA_BETA} (FIXED, v9 sweep winner)",
    "LACA g(Δτ)":          "mlp Linear(1,32)-Tanh-Linear(32,1), 97 params (FIXED; v10 form + capacity comparisons)",
    "LACA heads":          "num_heads = 4 (FIXED, conservative default kept after v11)",
    "Mamba backbone":      "2 blocks x d_state 16 x d_conv 2, expand 2, fused conv path — 874,496 params "
                           "(FIXED, v12 sweep winner)",
    "Aux loss weights":    "lambda_align = 0.1, lambda_cons = 0.1 (FIXED; both REAL in v13). No ordinal loss (v14 rejected)",
    "SEEDS":               "42 (reference, must reproduce), 123, 2024, 7, 99 — ensembles of the first 2/3/4/5",
    "Audio windowing":     "none (raw 1536-sample chunk -> rfft; no Hann)",
    "Audio compression":   "sqrt(magnitude)",
    "Audio normalization": "none",
    "Classification loss": "class-weighted cross-entropy (inverse TRAIN-set class frequency)",
    "Seeding":             "per seed: random / numpy / torch / cuda + cuDNN deterministic, reset at run start and "
                           "before model construction (v13 mechanism); mission split fixed at random.seed(42)",
    "Everything else":     "v13 unchanged (sensor path, split, 20 epochs, AdamW, eval)",
}
print("=" * 90)
print("  AEROSYNC-MAMBA v15 — 5-SEED VARIANCE + PROBABILITY ENSEMBLE")
print("=" * 90)
for _k, _v in PIPELINE_CONFIG.items():
    print(f"  • {_k:<20}: {_v}")
print("=" * 90)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

# ==============================================================================
# FIX 2: install the REAL mamba_ssm package (selective-scan SSM kernels).
#
# STRATEGY (in order):
#   1. PREBUILT WHEEL from the causal-conv1d / mamba-ssm GitHub releases, matched
#      to this exact Python / torch / CUDA / C++ ABI build. This is the ONLY
#      reliable path on Kaggle — building from source routinely hangs or OOMs
#      the kernel because Kaggle's toolchain doesn't line up with what these
#      packages' setup.py expects.
#   2. Capped from-source build (240s timeout, output captured) as a fallback
#      if no matching prebuilt wheel exists for this environment.
#   3. Pure-PyTorch fallback block (Cell 2) if both of the above fail — this is
#      the safety net that guarantees the notebook can NEVER hang/crash here,
#      but every mode in the frequency-bin sweep shares whichever backbone this
#      cell resolves to, so the 6-way comparison stays apples-to-apples either way.
# ==============================================================================
MAMBA_INSTALL_TIMEOUT_S = 240

def _detect_env_tags():
    """Python tag, torch major.minor, CUDA major.minor, C++11 ABI flag — the
    axes prebuilt causal-conv1d / mamba-ssm wheels are keyed on."""
    import sys
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    cuda_ver = torch.version.cuda  # e.g. "12.1", or None on CPU-only torch
    try:
        abi_flag = bool(torch._C._GLIBCXX_USE_CXX11_ABI)
    except Exception:
        abi_flag = bool(getattr(torch, "compiled_with_cxx11_abi", lambda: False)())
    return py_tag, torch_mm, cuda_ver, abi_flag

def _github_releases(repo, timeout_s=20):
    import urllib.request, json as _json
    url = f"https://api.github.com/repos/{repo}/releases"
    req = urllib.request.Request(url, headers={"User-Agent": "kaggle-notebook", "Accept": "application/vnd.github+json"})
    with urllib.request.urlopen(req, timeout=timeout_s) as resp:
        return _json.load(resp)

def _find_prebuilt_wheel_url(repo, py_tag, torch_mm, cuda_ver, abi_flag):
    """Best-matching .whl asset URL from `repo`'s GitHub releases, or None."""
    try:
        releases = _github_releases(repo)
    except Exception as e:
        print(f"    could not query github.com/{repo}/releases: {e!r}")
        return None

    abi_str = f"cxx11abi{'TRUE' if abi_flag else 'FALSE'}"
    cuda_major = cuda_ver.split(".")[0] if cuda_ver else None
    cuda_tag_exact = f"cu{cuda_ver.replace('.', '')}" if cuda_ver else None   # e.g. "cu121"
    exact, major_only = [], []

    for rel in releases:
        for asset in rel.get("assets", []):
            name = asset.get("name", "")
            if not name.endswith(".whl"):
                continue
            if py_tag not in name or abi_str not in name or f"torch{torch_mm}" not in name:
                continue
            if cuda_tag_exact and cuda_tag_exact in name:
                exact.append(asset["browser_download_url"])
            elif cuda_major and f"cu{cuda_major}" in name:
                major_only.append(asset["browser_download_url"])

    chosen = (exact or major_only)
    return chosen[0] if chosen else None

def _pip_run(args, timeout_s=MAMBA_INSTALL_TIMEOUT_S):
    """pip invocation that can NEVER hang or crash the kernel: hard timeout,
    output captured (not streamed — a runaway nvcc/ninja build can print
    megabytes), every exception caught."""
    import subprocess, sys
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + args
    try:
        proc = subprocess.run(cmd, timeout=timeout_s, capture_output=True, text=True, check=False)
        if proc.returncode != 0:
            tail = (proc.stderr or proc.stdout or "")[-500:]
            print(f"    ⚠️  pip install {args} failed (exit {proc.returncode}); tail:\n{tail}")
            return False
        return True
    except subprocess.TimeoutExpired:
        print(f"    ⚠️  pip install {args} exceeded {timeout_s}s — killed to avoid hanging/OOM-ing "
              f"the kernel; treating as failed.")
        return False
    except Exception as e:
        print(f"    ⚠️  pip install {args} raised {e!r}; treating as failed.")
        return False

def _install_via_prebuilt_wheel(pip_name, repo, py_tag, torch_mm, cuda_ver, abi_flag):
    url = _find_prebuilt_wheel_url(repo, py_tag, torch_mm, cuda_ver, abi_flag)
    if url is None:
        print(f"    no matching prebuilt wheel found for {pip_name} "
              f"(py={py_tag}, torch={torch_mm}, cuda={cuda_ver}, cxx11abi={abi_flag})")
        return False
    print(f"    found prebuilt wheel: {url}")
    return _pip_run([url, "--no-deps"])

_MAMBA_INSTALL_METHOD = "not attempted (no CUDA GPU)"

if torch.cuda.is_available():
    try:
        import mamba_ssm  # noqa: F401
        print("mamba_ssm already installed.")
        _MAMBA_INSTALL_METHOD = "already installed"
    except ImportError:
        py_tag, torch_mm, cuda_ver, abi_flag = _detect_env_tags()
        print(f"Environment: python={py_tag} | torch={torch_mm} | cuda={cuda_ver} | cxx11abi={abi_flag}")
        print("Installing real mamba_ssm + causal-conv1d (selective-scan SSM kernels)...")

        print("  [1/3] trying prebuilt wheels (no compilation — the reliable path on Kaggle)...")
        ok1 = _install_via_prebuilt_wheel("causal-conv1d", "Dao-AILab/causal-conv1d", py_tag, torch_mm, cuda_ver, abi_flag)
        ok2 = _install_via_prebuilt_wheel("mamba-ssm", "state-spaces/mamba", py_tag, torch_mm, cuda_ver, abi_flag) if ok1 else False

        if ok1 and ok2:
            _MAMBA_INSTALL_METHOD = "prebuilt wheel"
        else:
            print(f"  [2/3] no complete prebuilt-wheel match — falling back to a capped "
                  f"from-source build ({MAMBA_INSTALL_TIMEOUT_S}s timeout per package)...")
            # --no-build-isolation avoids re-resolving/reinstalling torch inside an isolated
            # build env, a common cause of causal-conv1d's build spinning forever on Kaggle.
            ok1b = _pip_run(["causal-conv1d>=1.2.0", "--no-build-isolation"])
            ok2b = _pip_run(["mamba-ssm", "--no-build-isolation"]) if ok1b else False
            _MAMBA_INSTALL_METHOD = "source build" if (ok1b and ok2b) else "failed"

        print(f"  [3/3] resolution: {_MAMBA_INSTALL_METHOD}")
        if _MAMBA_INSTALL_METHOD in ("prebuilt wheel", "source build"):
            print(f"✅ causal-conv1d + mamba-ssm installed via {_MAMBA_INSTALL_METHOD}.")
        else:
            print("⚠️  mamba_ssm install did not complete — the pure-PyTorch FALLBACK block will be "
                  "used (see Cell 2 / Cell 6 output). This does not affect the frequency-bin sweep: "
                  "every mode uses the same backbone, so the 6-way comparison is still apples-to-apples.")
else:
    print("⚠️  No CUDA GPU detected — mamba_ssm requires CUDA, so the pure-PyTorch FALLBACK block will be used (see Cell 2 / Cell 6 output).")


# ==============================================================================
# GLOBAL SEEDING — python `random`, numpy, torch CPU + every CUDA device, plus
# deterministic cuDNN. The mission split in Cell 4 keeps its own
# random.seed(42); this covers everything else (weight init, dropout, DataLoader
# shuffle order). set_all_seeds() is called again right before model
# construction for every seed of the multi-seed run (Cell 6), with that seed's value.
# Note: mamba_ssm's CUDA selective-scan backward uses atomic adds, so GPU runs
# can still differ in the last few decimal places even with identical seeds.
# ==============================================================================
def set_all_seeds(seed: int = SEED) -> int:
    """Reset python / numpy / torch (CPU + all CUDA devices) RNGs to `seed`."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    return seed

set_all_seeds(SEED)
print(f"✅ Seeds set -> random={SEED}, numpy={SEED}, torch={SEED}, cuda={SEED} (cuDNN deterministic)")

  AEROSYNC-MAMBA v15 — 5-SEED VARIANCE + PROBABILITY ENSEMBLE
  • Audio bin selection : strongest-64 (N=64 frequency bins, v6 winner; TRAINING audio only, frozen for val/test)
  • Audio time positions: Ta = 64 per window (FIXED, v7)
  • Sensor input        : raw 6 IMU channels (accel x/y/z, gyro x/y/z), Ts = 128 (FIXED; no derived magnitudes)
  • LACA beta           : temporal_bias_scale = 1.0 (FIXED, v9 sweep winner)
  • LACA g(Δτ)          : mlp Linear(1,32)-Tanh-Linear(32,1), 97 params (FIXED; v10 form + capacity comparisons)
  • LACA heads          : num_heads = 4 (FIXED, conservative default kept after v11)
  • Mamba backbone      : 2 blocks x d_state 16 x d_conv 2, expand 2, fused conv path — 874,496 params (FIXED, v12 sweep winner)
  • Aux loss weights    : lambda_align = 0.1, lambda_cons = 0.1 (FIXED; both REAL in v13). No ordinal loss (v14 rejected)
  • SEEDS               : 42 (reference, must reproduce), 123, 2024, 7, 99 — ensembles of the first 2/3/4/5
  • Audio windowing  

# Cell 2: Bi-Modal AeroSync-Mamba Architecture (HTT + LACA + Mamba) — mlp g(Δτ) 97 params, FIXED Mamba backbone 2 x d_state 16 x d_conv 2 (v12 winner)

In [2]:
# --- Fusion backbone resolution: REAL mamba_ssm (CUDA selective-scan) vs. pure-PyTorch FALLBACK ---
_MAMBA_CLASS = None
_FUSION_BACKBONE_REASON = None
try:
    if torch.cuda.is_available():
        from mamba_ssm import Mamba as _MAMBA_CLASS
        _FUSION_BACKBONE_REASON = "mamba_ssm.Mamba (REAL selective-scan SSM: hidden state h_t, learned A/B/C, discretized Δ) — CUDA GPU detected"
    else:
        _FUSION_BACKBONE_REASON = "FALLBACK (VectorizedSSMBlock, pure PyTorch, NO SSM recurrence) — no CUDA GPU detected; mamba_ssm requires CUDA"
except ImportError as _mamba_import_err:
    _FUSION_BACKBONE_REASON = f"FALLBACK (VectorizedSSMBlock, pure PyTorch, NO SSM recurrence) — mamba_ssm import failed: {_mamba_import_err}"

print(f"\n{'='*70}\nFUSION BACKBONE RESOLUTION: {_FUSION_BACKBONE_REASON}\n{'='*70}\n")


# --- 1. Continuous Temporal Positional Embedding & HTT ---
class ContinuousTimeEmbedding(nn.Module):
    def __init__(self, d_model: int, num_frequencies: int = 64):
        super().__init__()
        self.frequencies = nn.Parameter(
            torch.exp(torch.linspace(0, math.log(1000.0), num_frequencies)), requires_grad=False
        )
        self.mlp = nn.Sequential(
            nn.Linear(2 * num_frequencies, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, tau: torch.Tensor) -> torch.Tensor:
        if tau.dim() == 2:
            tau = tau.unsqueeze(-1)
        angles = tau * self.frequencies * 2.0 * math.pi
        fourier = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)
        return self.mlp(fourier)


class HeterogeneousTemporalTokenization(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.d_model = d_model
        # 0: Audio, 1: Sensor Telemetry
        self.modality_embeddings = nn.Embedding(2, d_model)
        self.time_embedding = ContinuousTimeEmbedding(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, z_bar: torch.Tensor, tau: torch.Tensor, modality_idx: int) -> torch.Tensor:
        mod_idx = torch.tensor(modality_idx, device=z_bar.device, dtype=torch.long)
        e_m = self.modality_embeddings(mod_idx).view(1, 1, self.d_model)
        phi_tau = self.time_embedding(tau)
        return self.norm(z_bar + e_m + phi_tau)


# --- 2. Bi-Modal LACA (Audio <-> Telemetry Alignment) ---
class PairwiseLACA(nn.Module):
    def __init__(self, d_model: int, num_heads: int = 4, beta: float = 1.0):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.beta = beta

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.lag_bias = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 1))

    def forward(self, H_m, H_n, tau_m, tau_n):
        B, T_m, _ = H_m.shape
        _, T_n, _ = H_n.shape
        Q = self.w_q(H_m).view(B, T_m, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.w_k(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        delta_tau = (tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)
        B_mn = self.lag_bias(delta_tau).squeeze(-1).unsqueeze(1)

        attn_weights = F.softmax(scores + self.beta * B_mn, dim=-1)
        out = torch.matmul(attn_weights, V).transpose(1, 2).contiguous().view(B, T_m, self.d_model)
        H_tilde_mn = self.out_proj(out)

        A_mn = attn_weights.mean(dim=1)
        tau_diff = tau_n.unsqueeze(1) - tau_m.unsqueeze(2)
        expected_lag = (A_mn * tau_diff).sum(dim=(1, 2)) / float(T_m)
        return H_tilde_mn, A_mn, expected_lag


class BiModalLACAFusion(nn.Module):
    def __init__(self, d_model: int, num_heads: int = 4, beta: float = 1.0):
        super().__init__()
        self.laca_as = PairwiseLACA(d_model, num_heads, beta)
        self.laca_sa = PairwiseLACA(d_model, num_heads, beta)
        self.gamma_as = nn.Parameter(torch.tensor(0.5))
        self.gamma_sa = nn.Parameter(torch.tensor(0.5))

    def forward(self, Ha, Hs, tau_a, tau_s):
        H_a_from_s, _, lag_as = self.laca_as(Ha, Hs, tau_a, tau_s)
        H_s_from_a, _, lag_sa = self.laca_sa(Hs, Ha, tau_s, tau_a)

        H_tilde_a = Ha + self.gamma_as * H_a_from_s
        H_tilde_s = Hs + self.gamma_sa * H_s_from_a
        return H_tilde_a, H_tilde_s, {"lag_as": lag_as, "lag_sa": lag_sa}


# --- 3a. FALLBACK: pure-PyTorch gated-conv block (NOT a real SSM) ---
# Used ONLY when mamba_ssm / CUDA is unavailable (see resolution printed
# above). This block has NO hidden state h_t, NO A/B/C matrices, and NO
# discretization step — it is a depthwise-conv + gated-MLP block, kept only
# as a last-resort fallback so the notebook still runs on CPU-only machines.
class VectorizedSSMBlock(nn.Module):
    def __init__(self, d_model: int, d_conv: int = 4, expand: int = 2):
        super().__init__()
        self.d_inner = int(expand * d_model)
        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner, kernel_size=d_conv,
            padding=d_conv - 1, groups=self.d_inner, bias=True
        )
        self.mixer = nn.Sequential(
            nn.Linear(self.d_inner, self.d_inner),
            nn.GELU(),
            nn.Linear(self.d_inner, self.d_inner)
        )
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, D = x.shape
        residual = x
        x_norm = self.norm(x)
        xz = self.in_proj(x_norm)
        x_branch, z_branch = xz.chunk(2, dim=-1)

        x_conv = self.conv1d(x_branch.transpose(1, 2))[:, :, :L].transpose(1, 2)
        x_conv = F.silu(x_conv)

        y = self.mixer(x_conv) * F.silu(z_branch)
        return residual + self.out_proj(y)


# --- 3b. REAL Mamba block (genuine selective-scan SSM) ---
# Thin wrapper around mamba_ssm.Mamba, which implements the actual S6
# recurrence (hidden state h_t, learned continuous A/B/C matrices, and a
# discretization step Δ), wrapped in the same pre-norm + residual pattern as
# the fallback block above so it is a drop-in replacement.
#
# v13 — the fusion backbone is FIXED at the v12 sweep winner (Stage A num_blocks,
# Stage B d_state, Stage C d_conv): 2 blocks x d_state 16 x d_conv 2, expand 2.
# d_conv=2 is inside the causal-conv1d kernel's supported widths (2..4), so
# mamba_ssm takes its natural fused path — exactly as the winning v12 run C1 did
# (v12's forced-PyTorch-conv control showed the implementation choice itself makes
# no difference). The sweep scaffolding (configurable num_blocks / d_state / d_conv,
# the forced-PyTorch-conv switch and its kernel-disabling context manager) is gone.
import sys

MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
EXPECTED_MAMBA_PARAMS = 874_496      # v12 Stage C winner C1: 2 blocks x 437,248
# The v5-v10 backbone (4 blocks, d_state 16, d_conv 4). Used ONLY to advance the torch RNG
# (see BiModalAeroSyncMamba.__init__) — it is never part of the model.
_V10_BACKBONE_FOR_RNG = (4, 16, 4)


def mamba_conv_path():
    """Which local-conv implementation mamba_ssm will use (informational)."""
    if _MAMBA_CLASS is None:
        return "native (fallback VectorizedSSMBlock nn.Conv1d)"
    if getattr(sys.modules[_MAMBA_CLASS.__module__], "causal_conv1d_fn", None) is not None:
        return "fused (causal-conv1d CUDA kernel)"
    return "pytorch nn.Conv1d (causal-conv1d not importable)"


class RealMambaBlock(nn.Module):
    def __init__(self, d_model: int, d_conv: int = MAMBA_D_CONV, expand: int = MAMBA_EXPAND,
                 d_state: int = MAMBA_D_STATE):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = _MAMBA_CLASS(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.mamba(self.norm(x))


def build_fusion_blocks(d_model, num_blocks, d_state, d_conv, expand=MAMBA_EXPAND):
    if _MAMBA_CLASS is not None:
        return nn.ModuleList([
            RealMambaBlock(d_model=d_model, d_conv=d_conv, expand=expand, d_state=d_state)
            for _ in range(num_blocks)
        ])
    # the fallback block has no SSM state, so d_state does not apply to it
    return nn.ModuleList([
        VectorizedSSMBlock(d_model=d_model, d_conv=d_conv, expand=expand) for _ in range(num_blocks)
    ])


# --- 4. Full Bi-Modal AeroSync-Mamba Model ---
class BiModalAeroSyncMamba(nn.Module):
    def __init__(
        self,
        d_model: int = 256,
        n_classes: int = 5,
        dropout: float = 0.15,
        temporal_bias_scale: float = 1.0,
        audio_in_dim: int = 64,
        sensor_in_dim: int = 6,
    ):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes

        # Sensor 1D-CNN encoder
        self.sensor_encoder = nn.Sequential(
            nn.Conv1d(sensor_in_dim, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, d_model, kernel_size=3, padding=1), nn.BatchNorm1d(d_model), nn.GELU(),
        )
        # Audio feature projector
        self.audio_encoder = nn.Sequential(
            nn.Linear(audio_in_dim, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Linear(d_model, d_model)
        )

        self.htt = HeterogeneousTemporalTokenization(d_model)
        self.laca = BiModalLACAFusion(d_model=d_model, num_heads=4, beta=temporal_bias_scale)
        self.sep_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        # --- FIX 2: use the REAL mamba_ssm block when available, otherwise the
        # pure-PyTorch fallback (see resolution printed at the top of this cell,
        # and the unmissable print in Cell 6 right after model construction). ---
        # v13: fixed backbone 2 x (d_state 16, d_conv 2). RNG alignment (NOT a sweep
        # option): the winning v12 run built this stack from the SAME torch RNG state
        # as the v5-v10 4-block stack and then restored the RNG to the state AFTER that
        # 4-block stack. Every module built after this point (pool_attn, classifier,
        # aux heads) and the whole training RNG stream (shuffle order, dropout masks)
        # therefore depend on it. To reproduce the v12 result bit-for-bit, a 4-block
        # stack is built and immediately DISCARDED purely to advance the RNG — it is
        # never registered as a submodule, has no parameters in the model and is never
        # run. Removing it would change every downstream initial weight.
        self.mamba_config = {"num_blocks": MAMBA_NUM_BLOCKS, "d_state": MAMBA_D_STATE,
                             "d_conv": MAMBA_D_CONV, "expand": MAMBA_EXPAND}
        _rng_pre = torch.get_rng_state()
        _rng_advance_only = build_fusion_blocks(d_model, *_V10_BACKBONE_FOR_RNG)
        del _rng_advance_only
        _rng_post = torch.get_rng_state()
        torch.set_rng_state(_rng_pre)
        self.mamba_blocks = build_fusion_blocks(d_model, MAMBA_NUM_BLOCKS, MAMBA_D_STATE, MAMBA_D_CONV)
        torch.set_rng_state(_rng_post)
        self.fusion_backbone_reason = _FUSION_BACKBONE_REASON

        self.pool_attn = nn.Sequential(nn.Linear(d_model, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, n_classes),
        )

        self.aux_a = nn.Linear(d_model, n_classes)
        self.aux_s = nn.Linear(d_model, n_classes)

    def forward(self, xa, xs, ta, ts):
        B = xa.shape[0]

        # 1. Modality Encoders
        za = self.audio_encoder(xa)
        zs = self.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)

        # 2. HTT (0: Audio, 1: Sensor)
        Ha = self.htt(za, ta, modality_idx=0)
        Hs = self.htt(zs, ts, modality_idx=1)

        # 3. Bi-Modal LACA Alignment
        H_tilde_a, H_tilde_s, aux = self.laca(Ha, Hs, ta, ts)
        za_p = H_tilde_a.mean(dim=1)
        zs_p = H_tilde_s.mean(dim=1)

        # 4. Cross-Modal Mamba Fusion: [ H_tilde_a ; [SEP] ; H_tilde_s ]
        sep = self.sep_token.expand(B, -1, -1)
        H_fused = torch.cat([H_tilde_a, sep, H_tilde_s], dim=1)

        F_seq = H_fused
        for blk in self.mamba_blocks:
            F_seq = blk(F_seq)

        weights = F.softmax(self.pool_attn(F_seq), dim=1)
        zf = (F_seq * weights).sum(dim=1)
        logits = self.classifier(zf)

        pooled_reps = {"z_a": za_p, "z_s": zs_p}
        aux_preds = {"y_a": self.aux_a(za_p), "y_s": self.aux_s(zs_p)}
        return logits, pooled_reps, aux_preds, aux

print(f"Bi-Modal AeroSync-Mamba model class defined successfully! | fixed Mamba backbone: {MAMBA_NUM_BLOCKS} blocks, "
      f"d_state={MAMBA_D_STATE}, d_conv={MAMBA_D_CONV}, expand={MAMBA_EXPAND} | conv path: {mamba_conv_path()}")


FUSION BACKBONE RESOLUTION: mamba_ssm.Mamba (REAL selective-scan SSM: hidden state h_t, learned A/B/C, discretized Δ) — CUDA GPU detected

Bi-Modal AeroSync-Mamba model class defined successfully! | fixed Mamba backbone: 2 blocks, d_state=16, d_conv=2, expand=2 | conv path: fused (causal-conv1d CUDA kernel)


# Cell 3: Bi-Modal Loss Function (Classification + lambda_align x Audio-IMU InfoNCE + lambda_cons x KL) — all raw components reported

In [3]:
# v13: L = L_cls + lambda_align * L_align + lambda_cons * L_cons. The two weights are set ONLY by
# the constructor arguments below (default 0.1 / 0.1 — the value every experiment v5-v12 used) and stored on
# the module as self.lambda_align / self.lambda_cons; forward() reads them from there. run_condition (Cell 6)
# passes each condition's weights in and verifies them on the constructed module.
class BiModalAeroSyncLoss(nn.Module):
    def __init__(self, lambda_align=0.1, lambda_cons=0.1, temperature=0.07, class_weights=None):
        super().__init__()
        self.lambda_align = lambda_align
        self.lambda_cons = lambda_cons
        self.temperature = temperature
        # --- FIX 4: class-weighted classification loss. `class_weights` is an
        # inverse-frequency tensor computed from REAL training-set class counts
        # in Cell 6 (printed there for verification), not hardcoded here. ---
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def _infonce(self, z1, z2):
        z1_norm = F.normalize(z1, p=2, dim=-1)
        z2_norm = F.normalize(z2, p=2, dim=-1)
        sim = torch.matmul(z1_norm, z2_norm.t()) / self.temperature
        labels = torch.arange(z1.shape[0], device=z1.device)
        return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.t(), labels))

    def forward(self, logits, targets, pooled, aux_preds):
        # 1. Fault Classification Loss
        loss_cls = self.ce(logits, targets)

        # 2. Audio-Sensor Contrastive Alignment Loss
        loss_align = self._infonce(pooled["z_a"], pooled["z_s"])

        # 3. Auxiliary Consistency Loss
        p_fused = F.softmax(logits.detach(), dim=-1)
        loss_cons = (
            F.kl_div(F.log_softmax(aux_preds["y_a"], dim=-1), p_fused, reduction="batchmean") +
            F.kl_div(F.log_softmax(aux_preds["y_s"], dim=-1), p_fused, reduction="batchmean")
        ) / 2.0

        total = loss_cls + self.lambda_align * loss_align + self.lambda_cons * loss_cons
        # v13: every RAW component is reported (loss_cons was previously not), so a term whose weight is
        # 0.0 can still be seen computing a sensible value even though it adds nothing to `total`.
        return total, {"loss_cls": float(loss_cls.item()), "loss_align": float(loss_align.item()),
                       "loss_cons": float(loss_cons.item())}

def compute_all_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    try:
        auroc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except Exception:
        auroc = float("nan")
    return {"accuracy": acc, "macro_f1": f1, "balanced_accuracy": bal_acc, "macro_auroc": auroc}

print("Loss & Metrics helper ready!")

Loss & Metrics helper ready!


# Cell 4 (Self-Downloading & Preprocessing): Real TII UAV Dataset

In [4]:
import os
import json
import random
import numpy as np
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
import torch

DATASET_BASE = Path("/kaggle/working/tii_uav_dataset")
DATASET_ROOT = DATASET_BASE / "Dataset"
CACHE_DIR = Path("/kaggle/working/preprocessed_bimodal_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# 1. Automatically clone the official repository if not present
if not DATASET_ROOT.exists():
    print("📥 Cloning official TII UAV Realistic Fault Dataset from GitHub...")
    !git clone --depth 1 https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git /kaggle/working/tii_uav_dataset
    print("✅ Dataset cloned successfully!")

_chunk_duration_log = []
_fft_shape_printed = False  # verification: print the FFT feature shape once

# Three different sizes — kept as separate named constants so they are never confused:
N_AUDIO_BINS = 64             # audio FREQUENCY axis: strongest-64 bins = feature dim per time step (FIXED, v6 winner)
TA_AUDIO = 64                 # audio TIME axis: time positions per window (FIXED, v7)
TS_SENSOR = 128               # sensor TIME axis: time positions per window (FIXED, unchanged from v6)
_audio_chunks_per_window = []  # verification: real audio chunks available in each window (vs Ta)
_MISSION_CACHE = {}  # mission_dir -> parse_real_mission(...) output, parsed ONCE and reused for every N

# 2. Real Flight Mission Parser (6-Axis IMU + Audio)
def parse_real_mission(mission_dir: Path):
    sensor_matches = list(mission_dir.glob("*SensorCombined*.jsonl"))
    audio_matches  = list(mission_dir.glob("*AudioBuffer*.jsonl"))
    if not sensor_matches or not audio_matches:
        return None

    # 2.1 Parse Real IMU
    sensor_times, accel_data, gyro_data = [], [], []
    with open(sensor_matches[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            try:
                rec = json.loads(line)
                t_sec = float(rec["timestamp"]) / 1e6
                sensor_times.append(t_sec)
                accel_data.append(rec.get("accelerometer_m_s2", [0.0, 0.0, 9.81]))
                gyro_data.append(rec.get("gyro_rad", [0.0, 0.0, 0.0]))
            except Exception:
                continue

    if len(sensor_times) < 50: return None
    sensor_times = np.array(sensor_times, dtype=np.float64)
    sensor_times = sensor_times - sensor_times[0]
    telemetry = np.concatenate([np.array(accel_data, dtype=np.float32), np.array(gyro_data, dtype=np.float32)], axis=-1)

    # 2.2 Parse Real Audio
    audio_chunks = []
    with open(audio_matches[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            try:
                rec = json.loads(line)
                samples = rec.get("audio_data", rec.get("buffer", rec.get("data", [])))
                if len(samples) > 0: audio_chunks.append(samples)
            except Exception:
                continue

    if len(audio_chunks) < 10: return None
    audio_chunks = np.array(audio_chunks, dtype=np.float32)

    # --- FIX: derive REAL per-chunk audio timestamps ---
    # `recording_started_time` is a constant repeated on every line (not a
    # real per-chunk timestamp), and the previous fixed 0.10s/chunk guess
    # doesn't match the data (empirically ~17-19ms/chunk). Instead, derive
    # chunk duration from this mission's REAL, IMU-verified flight duration,
    # since audio and IMU are recorded within the same continuous rosbag
    # session.
    mission_duration_s = sensor_times[-1] - sensor_times[0]  # already zero-based
    n_chunks = len(audio_chunks)
    chunk_duration_s = mission_duration_s / n_chunks
    audio_times = np.arange(n_chunks, dtype=np.float64) * chunk_duration_s

    global _chunk_duration_log
    _chunk_duration_log.append(chunk_duration_s)

    return {
        "sensor_times": sensor_times,
        "telemetry": telemetry,
        "audio_times": audio_times,
        "audio_chunks": audio_chunks,
    }


def get_mission(mission_dir: Path):
    """parse_real_mission, memoized: every mission's JSONL is parsed once and the
    same arrays are reused for every N (identical to re-parsing each time)."""
    if mission_dir not in _MISSION_CACHE:
        _MISSION_CACHE[mission_dir] = parse_real_mission(mission_dir)
    return _MISSION_CACHE[mission_dir]


# 2.3 Strongest-N FFT bin selection (TRAINING-SET audio only, unsupervised, leakage-safe)
def select_strongest_audio_bins(train_mission_dirs, n_bins):
    """
    Selects the `n_bins` FFT magnitude bins with the highest mean magnitude,
    computed ONLY from training-set audio (raw waveform energy — never class
    labels). Called once per N, before any train/val/test window slicing for
    that N, so the exact same frozen indices are reused identically across all
    three splits.

    Identical criterion to the v5 strongest-64 selection; only the budget N is
    a parameter now. Returns (sorted bin indices, data-derived bin width in Hz
    or None).
    """
    print(f"[STRONGEST-{n_bins}] Computing bin-selection statistics from TRAINING-SET audio only "
          f"({len(train_mission_dirs)} training missions) — BEFORE any train/validation/test "
          "window preprocessing.")

    bin_magnitude_sum = None
    bin_count = 0
    fft_len_ref = None
    local_chunk_durations = []

    for m_dir in tqdm(train_mission_dirs, desc=f"Bin selection N={n_bins} (train-only audio scan)", leave=False):
        mission_data = get_mission(m_dir)
        if mission_data is None:
            continue
        audio = mission_data["audio_chunks"]
        a_times = mission_data["audio_times"]
        if len(a_times) > 1:
            local_chunk_durations.append((a_times[-1] - a_times[0]) / (len(a_times) - 1))

        mag = np.abs(np.fft.rfft(audio, axis=1))  # (n_chunks, fft_len), unsupervised energy
        if fft_len_ref is None:
            fft_len_ref = mag.shape[1]
        min_len = min(fft_len_ref, mag.shape[1])
        mag = mag[:, :min_len]
        if bin_magnitude_sum is not None and len(bin_magnitude_sum) > min_len:
            bin_magnitude_sum = bin_magnitude_sum[:min_len]
        fft_len_ref = min_len

        chunk_sum = mag.sum(axis=0)
        bin_magnitude_sum = chunk_sum if bin_magnitude_sum is None else bin_magnitude_sum + chunk_sum
        bin_count += mag.shape[0]

    assert bin_magnitude_sum is not None and bin_count > 0, (
        f"STRONGEST-{n_bins} FAILED: no training-set audio chunks were available for bin selection."
    )

    mean_magnitude = bin_magnitude_sum / bin_count  # unsupervised magnitude/energy criterion
    top_bins = np.sort(np.argsort(mean_magnitude)[::-1][:n_bins]).astype(int)

    assert len(top_bins) == n_bins, (
        f"STRONGEST-{n_bins} FAILED: selected {len(top_bins)} bins, expected {n_bins}"
    )

    # --- Hz labeling ---
    # This notebook does not document a raw microphone sample rate anywhere,
    # so we estimate an effective sample rate from quantities the notebook
    # already derives from real data: samples-per-audio-buffer (inverted from
    # the rFFT length) and the real per-mission inter-buffer timing computed
    # above (same derivation `parse_real_mission` already uses for
    # `chunk_duration_s`). This is a data-derived estimate used only for
    # human-readable labeling below — it never affects which bins are
    # selected or how features are built.
    samples_per_buffer = fft_len_ref * 2 - 2
    est_sample_rate_hz = (
        samples_per_buffer / np.mean(local_chunk_durations) if local_chunk_durations else None
    )
    bin_width_hz = est_sample_rate_hz / samples_per_buffer if est_sample_rate_hz else None

    print(f"[STRONGEST-{n_bins}] Selected {n_bins} bins from {bin_count} training-set audio chunks "
          f"across {len(train_mission_dirs)} missions (FFT length = {fft_len_ref}).")
    print(f"[STRONGEST-{n_bins}] Selected bin indices (ascending): {top_bins.tolist()}")
    if bin_width_hz:
        print(f"[STRONGEST-{n_bins}] Estimated audio sample rate ≈ {est_sample_rate_hz:.1f} Hz "
              "(derived from audio buffer length and real inter-buffer timing across training "
              "missions; the dataset does not document a mic sample rate directly, so this is "
              "a data-derived estimate for labeling only, not a hardcoded spec).")
        # contiguous runs of selected bins -> approx. frequency bands
        runs, start = [], top_bins[0]
        for prev, cur in zip(top_bins[:-1], top_bins[1:]):
            if cur != prev + 1:
                runs.append((start, prev))
                start = cur
        runs.append((start, top_bins[-1]))
        print(f"[STRONGEST-{n_bins}] Selected bins as {len(runs)} contiguous band(s) -> approx. frequency (Hz):")
        for lo, hi in runs:
            print(f"    bins {lo:3d}-{hi:3d} ({hi - lo + 1:3d} bins)  ->  {lo * bin_width_hz:8.1f} - {hi * bin_width_hz:8.1f} Hz")
    else:
        print(f"[STRONGEST-{n_bins}] Could not estimate a sample rate (insufficient timing data); "
              "printing bin indices only.")

    return top_bins, bin_width_hz


# 2.4 Audio feature extraction — the single, permanent path (v5), width = N:
#     raw chunk -> rfft (no window) -> |.| -> frozen strongest-N bins -> sqrt
#     (no normalization).
def compute_audio_features(audio_chunks, bin_indices):
    """
    audio_chunks : (n_chunks, chunk_len) raw waveform chunks of one mission
    bin_indices  : frozen strongest-N bin indices from select_strongest_audio_bins
                   (computed once per N from TRAINING audio; never recomputed here)
    returns      : (n_chunks, N) float32 = sqrt(|rfft|) at those bins, N = len(bin_indices)
    """
    assert bin_indices is not None and len(bin_indices) > 0, (
        "FEATURES FAILED: strongest-N bin indices must be computed from training audio "
        "(select_strongest_audio_bins) before any feature extraction."
    )
    n_bins = len(bin_indices)

    audio_fft_mag = np.abs(np.fft.rfft(audio_chunks, axis=1))  # (n_chunks, chunk_len//2 + 1), REAL FFT
    n_chunks_mission = audio_fft_mag.shape[0]
    n_bins_available = audio_fft_mag.shape[1]
    valid_idx = bin_indices[bin_indices < n_bins_available]

    a_feats = np.zeros((n_chunks_mission, n_bins), dtype=np.float32)
    a_feats[:, :len(valid_idx)] = np.sqrt(audio_fft_mag[:, valid_idx]).astype(np.float32)

    assert a_feats.shape == (n_chunks_mission, n_bins), (
        f"FEATURES FAILED: audio feature array shape is {a_feats.shape}, "
        f"expected (n_chunks={n_chunks_mission}, N={n_bins})"
    )
    return a_feats


# 3. Slicing into 2.56s Diagnostic Samples
def slice_real_bimodal(mission_data, class_label, bin_indices, Ta, window_sec=2.56, step_sec=1.28, Ts=128):
    # Ta = audio TIME positions per window (required, no default); Ts = sensor TIME positions (unchanged).
    # The frequency width of every audio time step is n_bins = len(bin_indices), independent of Ta.
    global _fft_shape_printed
    s_times = mission_data["sensor_times"]
    telemetry = mission_data["telemetry"]
    a_times = mission_data["audio_times"]
    audio = mission_data["audio_chunks"]

    assert bin_indices is not None and len(bin_indices) > 0, (
        "SLICE FAILED: strongest-N bin indices must be computed from training audio "
        "(select_strongest_audio_bins) before any window preprocessing."
    )

    # --- strongest-N audio features: FFT magnitude spectrum, frozen top-N bins ---
    # Previously: the first 64 FFT bins (low frequencies only) were used
    # unconditionally for every split. Now: the 64 bins with the highest mean
    # magnitude in TRAINING-set audio (computed once in
    # select_strongest_audio_bins, unsupervised, leakage-safe) are frozen and
    # reused identically for train, validation, and test.
    # FFT (no window) -> frozen strongest-N bins -> sqrt, via compute_audio_features (2.4).
    n_bins = len(bin_indices)
    a_feats = compute_audio_features(audio, bin_indices)
    n_chunks_mission = audio.shape[0]

    assert a_feats.shape == (n_chunks_mission, n_bins), (
        f"STRONGEST-{n_bins} FAILED: audio feature array shape is {a_feats.shape}, "
        f"expected (n_chunks={n_chunks_mission}, {n_bins})"
    )

    if not _fft_shape_printed:
        print(f"[STRONGEST-{n_bins} VERIFICATION] audio chunk shape (raw waveform): {audio.shape} "
              f"| np.fft.rfft output shape: {(audio.shape[0], audio.shape[1] // 2 + 1)} "
              f"| final a_feats (sqrt-compressed, frozen strongest-{n_bins} bins) shape: {a_feats.shape}")
        print(f"[STRONGEST-{n_bins} VERIFICATION] Confirmed: features were produced via np.fft.rfft "
              f"(frequency-domain magnitude spectrum) using the frozen strongest-{n_bins} bin indices, "
              f"NOT the first {n_bins} bins.")
        _fft_shape_printed = True

    max_time = min(s_times[-1], a_times[-1])
    samples = []
    t_start = 0.0

    while t_start + window_sec <= max_time:
        t_end = t_start + window_sec
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < 10:
            t_start += step_sec
            continue

        # --- FIX 1: REAL per-window sensor timestamps ---
        # Previously: target_s_t = np.linspace(t_start, t_end, Ts) — a synthetic
        # uniform grid that is IDENTICAL for every window (window_sec is fixed),
        # so nothing real ever reached the model downstream. Now: pick Ts
        # evenly-spaced INDICES into the actual raw IMU sample timestamps that
        # fall inside this window. Real IMU logging is not perfectly uniform, so
        # these timestamps genuinely vary window-to-window and mission-to-mission.
        s_idx_in_window = np.where(s_mask)[0]
        real_s_pos = np.linspace(0, len(s_idx_in_window) - 1, Ts).round().astype(int)
        real_s_idx = s_idx_in_window[real_s_pos]
        target_s_t = s_times[real_s_idx]  # REAL raw sensor timestamps (seconds, mission-relative)

        s_window = np.zeros((Ts, 6), dtype=np.float32)
        for c in range(6):
            s_window[:, c] = np.interp(target_s_t, s_times[s_mask], telemetry[s_mask, c])

        # --- FIX 1: REAL per-window audio timestamps ---
        # audio_times itself is a per-mission uniform grid derived from the
        # mission's real chunk_duration_s (see parse_real_mission above), which
        # genuinely varies mission-to-mission (see the chunk-duration min/max
        # printed at the end of this cell). We pick real indices from it within
        # this window rather than an independent synthetic linspace.
        a_mask = (a_times >= t_start) & (a_times <= t_end)
        _audio_chunks_per_window.append(int(a_mask.sum()))
        if a_mask.sum() >= 2:
            a_idx_in_window = np.where(a_mask)[0]
            real_a_pos = np.linspace(0, len(a_idx_in_window) - 1, Ta).round().astype(int)
            real_a_idx = a_idx_in_window[real_a_pos]
            target_a_t = a_times[real_a_idx]  # REAL per-mission audio timestamps
        else:
            # Fallback only for the rare window with too few real audio samples
            target_a_t = np.linspace(t_start, t_end, Ta)

        a_window = np.zeros((Ta, n_bins), dtype=np.float32)  # (TIME positions, FREQUENCY bins)
        for c in range(n_bins):
            a_window[:, c] = np.interp(target_a_t, a_times, a_feats[:, c], left=0.0, right=0.0)

        assert a_window.shape == (Ta, n_bins), (
            f"STRONGEST-{n_bins} FAILED: window audio feature shape is {a_window.shape}, "
            f"expected ({Ta}, {n_bins})"
        )

        samples.append({
            "x_a": a_window,
            "x_s": s_window,
            "label": int(class_label),
            "tau_a": (target_a_t - t_start).astype(np.float32),  # REAL, zero-based, seconds
            "tau_s": (target_s_t - t_start).astype(np.float32),  # REAL, zero-based, seconds
            "window_sec": np.float32(window_sec),
        })
        t_start += step_sec

    return samples

# --- VERIFICATION: no leftover mode-switching in the audio feature path ---
import inspect
_feat_params = list(inspect.signature(compute_audio_features).parameters)
_slice_params = list(inspect.signature(slice_real_bimodal).parameters)
_switch_names = ("window_fn", "compression", "norm_stats", "normalization", "audio_mode", "mode")
_switch_params = [p for p in _feat_params + _slice_params if p in _switch_names]
try:
    _feat_src = inspect.getsource(compute_audio_features)
    _dead_branches = [t for t in ("hanning", "log1p", "log10", "zscore", "window_fn", "compression ==",
                                  "norm_stats", "elif") if t in _feat_src]
except (OSError, TypeError):
    _feat_src, _dead_branches = None, []
_stale_globals = [n for n in ("AUDIO_WINDOW_FNS", "AUDIO_COMPRESSIONS", "LOG_DB_EPS", "get_audio_window",
                              "compute_train_norm_stats", "N_BUDGETS", "TA_VALUES", "SENSOR_CHANNEL_CONFIGS",
                              "DERIVED_CHANNEL_SOURCES") if n in globals()]

assert _feat_params == ["audio_chunks", "bin_indices"], f"compute_audio_features has unexpected parameters: {_feat_params}"
assert not _switch_params, f"mode-switching parameters remain: {_switch_params}"
assert not _dead_branches, f"dead condition branches remain in compute_audio_features: {_dead_branches}"
print(f"[NO-MODE-SWITCH CHECK] compute_audio_features{tuple(_feat_params)} | slice_real_bimodal has no "
      f"window/compression/normalization parameters | source scan for hann/log1p/log_db/zscore/elif "
      f"branches: {'clean' if _feat_src is not None else 'source unavailable (signature check only)'} ✅")
if _stale_globals:
    print(f"⚠️  stale globals from an older notebook are still in this kernel: {_stale_globals} "
          f"(unused by v9 — restart the kernel for a clean run)")
else:
    print("[NO-MODE-SWITCH CHECK] no experiment globals (window / compression / normalization options) and no leftover N / Ta / sensor-feature globals defined ✅")

# 4. Mission-Level Stratification across all 99 Flights
class_missions = defaultdict(list)
for c_dir in sorted(DATASET_ROOT.iterdir()):
    if not c_dir.is_dir() or c_dir.name.startswith("."): continue
    try: c_label = int(c_dir.name)
    except ValueError: continue
    for m_dir in sorted(c_dir.iterdir()):
        if m_dir.is_dir() and not m_dir.name.startswith("."):
            class_missions[c_label].append(m_dir)

random.seed(42)

train_mission_map, val_mission_map, test_mission_map = {}, {}, {}
for c_label in sorted(class_missions.keys()):
    missions = class_missions[c_label]
    random.shuffle(missions)
    n_train = max(1, int(len(missions) * 0.70))
    n_val = max(1, int(len(missions) * 0.15))

    train_mission_map[c_label] = missions[:n_train]
    val_mission_map[c_label]   = missions[n_train:n_train + n_val]
    test_mission_map[c_label]  = missions[n_train + n_val:]

    print(f"Class {c_label} ({len(missions)} flights) -> "
          f"Train: {len(train_mission_map[c_label])}, "
          f"Val: {len(val_mission_map[c_label])}, "
          f"Test: {len(test_mission_map[c_label])}")

# --- 5. Strongest-64 bins: selected ONCE from TRAINING missions only, frozen ---
# The FREQUENCY axis does not depend on the audio TIME axis (Ta), so the same
# frozen indices are reused for every Ta condition and for train/val/test.
all_train_missions = [m for missions in train_mission_map.values() for m in missions]
_heldout_missions = {m for mp in (val_mission_map, test_mission_map) for ms in mp.values() for m in ms}
assert not (set(all_train_missions) & _heldout_missions), (
    "LEAKAGE: training missions overlap validation/test missions")

SELECTED_BIN_INDICES, AUDIO_BIN_WIDTH_HZ = select_strongest_audio_bins(all_train_missions, n_bins=N_AUDIO_BINS)
_bin_fingerprint_at_selection = (SELECTED_BIN_INDICES[:5].tolist(), SELECTED_BIN_INDICES[-5:].tolist())
print(f"[STRONGEST-{N_AUDIO_BINS}] frozen once from {len(all_train_missions)} TRAINING missions "
      f"(0 of {len(_heldout_missions)} validation/test missions used); reused unchanged for every Ta.")


# --- 6. Per-Ta preprocessing (called once per Ta by run_experiment in Cell 6) ---
def preprocess_for_ta(Ta):
    """Train/val/test windows with Ta audio time positions per window, using the
    frozen strongest-64 bins. Sensor side (TS_SENSOR positions, 6 channels) is
    built by exactly the same code as v6 and does not depend on Ta."""
    global _fft_shape_printed
    _fft_shape_printed = False
    _audio_chunks_per_window.clear()

    split_maps = [("TRAIN", train_mission_map), ("VAL", val_mission_map), ("TEST", test_mission_map)]
    split_samples = {}
    _fp_train = None
    for split_label, mission_map in split_maps:
        note = "(frozen strongest-64 bins)" if split_label == "TRAIN" else "(same frozen bin indices — NOT recomputed)"
        print(f"\n--- Preprocessing {split_label} split, Ta={Ta} {note} ---")
        samples = []
        for c_label in sorted(class_missions.keys()):
            for m in tqdm(mission_map[c_label], desc=f"{split_label.capitalize()} Class {c_label} (Ta={Ta})", leave=False):
                p = get_mission(m)
                if p:
                    samples.extend(slice_real_bimodal(p, c_label, SELECTED_BIN_INDICES, Ta=Ta, Ts=TS_SENSOR))
        split_samples[split_label] = samples

        _fp = (SELECTED_BIN_INDICES[:5].tolist(), SELECTED_BIN_INDICES[-5:].tolist())
        if split_label == "TRAIN":
            _fp_train = _fp
            print(f"[STRONGEST-{N_AUDIO_BINS}] Frozen bin indices applied @ TRAIN (first5, last5): {_fp} "
                  f"| matches selection: {_fp == _bin_fingerprint_at_selection}")
        else:
            print(f"[STRONGEST-{N_AUDIO_BINS}] Frozen bin indices applied @ {split_label:<5} (first5, last5): {_fp} "
                  f"| matches TRAIN: {_fp == _fp_train}")

    # --- Shape verification across all three splits:
    #     audio  (n_windows, Ta, N_AUDIO_BINS)  = (windows, TIME, FREQUENCY)
    #     sensor (n_windows, TS_SENSOR, 6)      — independent of Ta
    sensor_fingerprint = {}
    for split_label, samples in split_samples.items():
        x_a_all = np.stack([s["x_a"] for s in samples])
        tau_a_all = np.stack([s["tau_a"] for s in samples])
        x_s_all = np.stack([s["x_s"] for s in samples])
        tau_s_all = np.stack([s["tau_s"] for s in samples])
        assert x_a_all.shape == (len(samples), Ta, N_AUDIO_BINS), (
            f"Ta={Ta} FAILED: {split_label} audio array is {x_a_all.shape}, "
            f"expected (n_windows={len(samples)}, Ta={Ta}, N_AUDIO_BINS={N_AUDIO_BINS})")
        assert tau_a_all.shape == (len(samples), Ta), f"Ta={Ta} FAILED: {split_label} tau_a is {tau_a_all.shape}"
        assert x_s_all.shape == (len(samples), TS_SENSOR, 6), (
            f"Ta={Ta} FAILED: {split_label} sensor array is {x_s_all.shape}, expected (n_windows, {TS_SENSOR}, 6)")
        assert tau_s_all.shape == (len(samples), TS_SENSOR), f"Ta={Ta} FAILED: {split_label} tau_s is {tau_s_all.shape}"
        sensor_fingerprint[split_label] = (len(samples), round(float(x_s_all.astype(np.float64).sum()), 3),
                                           round(float(tau_s_all.astype(np.float64).sum()), 3),
                                           tuple(np.bincount([s["label"] for s in samples], minlength=5).tolist()))
        print(f"[SHAPE CHECK] {split_label:<5}: audio {x_a_all.shape} = (n_windows, Ta={Ta}, bins={N_AUDIO_BINS}) ✅ | "
              f"sensor {x_s_all.shape} = (n_windows, Ts={TS_SENSOR}, 6) ✅")
        del x_a_all, tau_a_all, x_s_all, tau_s_all

    cpw = np.array(_audio_chunks_per_window)
    n_short = int((cpw < Ta).sum())
    print(f"[AUDIO RESOLUTION] real audio chunks inside a {2.56:.2f}s window: min {cpw.min()} | median "
          f"{int(np.median(cpw))} | max {cpw.max()} -> Ta={Ta} "
          + ("sub-samples the real chunks in every window." if n_short == 0 else
             f"exceeds the available chunks in {n_short}/{len(cpw)} windows ({n_short / len(cpw) * 100:.1f}%), "
             "where some real chunk positions are repeated (no synthetic values are created)."))

    print("\n" + "=" * 60)
    print(f"     100% REAL PREPROCESSING COMPLETE (Ta={Ta}, ZERO SYNTHETIC DATA)   ")
    if _chunk_duration_log:
        _arr = np.array(_chunk_duration_log)
        print(f"Audio chunk duration check -> mean: {_arr.mean()*1000:.2f} ms | min: {_arr.min()*1000:.2f} ms | max: {_arr.max()*1000:.2f} ms")
    print("=" * 60)
    print(f"  • Total Train Diagnostic Windows : {len(split_samples['TRAIN'])}")
    print(f"  • Total Val Diagnostic Windows   : {len(split_samples['VAL'])}")
    print(f"  • Total Test Diagnostic Windows  : {len(split_samples['TEST'])}")
    print("=" * 60)

    out_dir = CACHE_DIR / f"Ta{Ta}"
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = {"train": out_dir / "bimodal_train.pt", "val": out_dir / "bimodal_val.pt", "test": out_dir / "bimodal_test.pt"}
    torch.save(split_samples["TRAIN"], paths["train"])
    torch.save(split_samples["VAL"], paths["val"])
    torch.save(split_samples["TEST"], paths["test"])
    print(f"✅ Cached real bimodal dataset for Ta={Ta} in {out_dir}")

    return {
        "paths": paths,
        "n_windows": (len(split_samples["TRAIN"]), len(split_samples["VAL"]), len(split_samples["TEST"])),
        "sensor_fingerprint": sensor_fingerprint,
        "chunks_per_window": (int(cpw.min()), int(np.median(cpw)), int(cpw.max())),
        "pct_windows_ta_exceeds_chunks": round(n_short / len(cpw) * 100, 2),
    }

print(f"\n✅ Preprocessing helpers ready — split + strongest-{N_AUDIO_BINS} bins fixed above; window "
      f"preprocessing (Ta = {TA_AUDIO}, Ts = {TS_SENSOR}) is built once in Cell 6 and reused for every beta.")

📥 Cloning official TII UAV Realistic Fault Dataset from GitHub...
Cloning into '/kaggle/working/tii_uav_dataset'...
remote: Enumerating objects: 324, done.
remote: Counting objects: 100% (324/324), done.
remote: Compressing objects: 100% (213/213), done.
remote: Total 324 (delta 111), reused 321 (delta 111), pack-reused 0 (from 0)
Receiving objects: 100% (324/324), 1.15 GiB | 27.45 MiB/s, done.
Resolving deltas: 100% (111/111), done.
Updating files: 100% (217/217), done.
✅ Dataset cloned successfully!
[NO-MODE-SWITCH CHECK] compute_audio_features('audio_chunks', 'bin_indices') | slice_real_bimodal has no window/compression/normalization parameters | source scan for hann/log1p/log_db/zscore/elif branches: clean ✅
[NO-MODE-SWITCH CHECK] no experiment globals (window / compression / normalization options) and no leftover N / Ta / sensor-feature globals defined ✅
Class 0 (19 flights) -> Train: 13, Val: 2, Test: 4
Class 1 (20 flights) -> Train: 14, Val: 3, Test: 3
Class 2 (20 flights) -> Tr

Bin selection N=64 (train-only audio scan):   0%|          | 0/69 [00:00<?, ?it/s]

[STRONGEST-64] Selected 64 bins from 356506 training-set audio chunks across 69 missions (FFT length = 769).
[STRONGEST-64] Selected bin indices (ascending): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760, 761, 762, 763, 764, 765, 766]
[STRONGEST-64] Estimated audio sample rate ≈ 84695.4 Hz (derived from audio buffer length and real inter-buffer timing across training missions; the dataset does not document a mic sample rate directly, so this is a data-derived estimate for labeling only, not a hardcoded spec).
[STRONGEST-64] Selected bins as 7 contiguous band(s) -> approx. frequency (Hz):
    bins   1- 11 ( 11 bins)  ->      55.1 -    606.5 Hz
    bins 245-255 ( 11 bins)  ->   13509.4 -  14060.8 Hz
    bins 258-267 ( 10 bins)  ->   14226.2 -  14722.4 Hz
    bins 501-

# Cell 5: DataLoader (weights_only=False for PyTorch 2.6+)

In [5]:
class RealBiModalDataset(Dataset):
    def __init__(self, pt_path: Path, mean=None, std=None):
        self.samples = torch.load(pt_path, weights_only=False)
        self.Ta = int(self.samples[0]["x_a"].shape[0])  # audio TIME positions per window
        self.Ts = 128

        if mean is None or std is None:
            all_s = np.concatenate([s["x_s"] for s in self.samples], axis=0)
            self.mean = all_s.mean(axis=0, keepdims=True)
            self.std = all_s.std(axis=0, keepdims=True) + 1e-6
        else:
            self.mean = mean
            self.std = std

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        norm_sensor = (s["x_s"] - self.mean) / self.std

        # --- FIX 1: use the REAL per-window timestamps saved during
        # preprocessing (Cell 4), normalized by that window's real duration,
        # instead of a synthetic torch.linspace(0.0, 1.0, ...) that was
        # identical for every sample regardless of the underlying data.
        window_sec = float(s["window_sec"])
        tau_a = torch.tensor(s["tau_a"], dtype=torch.float32) / window_sec
        tau_s = torch.tensor(s["tau_s"], dtype=torch.float32) / window_sec

        return {
            "x_a": torch.tensor(s["x_a"], dtype=torch.float32),
            "x_s": torch.tensor(norm_sensor, dtype=torch.float32),
            "tau_a": tau_a,
            "tau_s": tau_s,
            "label": torch.tensor(s["label"], dtype=torch.long),
        }

batch_size = 32

def build_dataloaders(paths, batch_size=batch_size):
    """Datasets + loaders (same settings as v7; identical for every beta)."""
    train_ds = RealBiModalDataset(paths["train"])
    val_ds   = RealBiModalDataset(paths["val"], mean=train_ds.mean, std=train_ds.std)
    test_ds  = RealBiModalDataset(paths["test"], mean=train_ds.mean, std=train_ds.std)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=2)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=2)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=2)

    print(f"DataLoaders ready -> Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)} windows")
    return train_ds, val_ds, test_ds, train_loader, val_loader, test_loader

print("RealBiModalDataset / build_dataloaders ready (DataLoaders are built per beta in Cell 6).")

RealBiModalDataset / build_dataloaders ready (DataLoaders are built per beta in Cell 6).


In [6]:
# ==============================================================================
# FIX 1 VERIFICATION HOOK — confirm tau_a / tau_s carry REAL, per-sample-varying
# timestamps, not the old synthetic torch.linspace(0.0, 1.0, ...) that was
# identical for every sample regardless of the underlying data.
# ==============================================================================
def verify_real_taus(train_ds):
    print("=" * 70)
    print("VERIFICATION HOOK — FIX 1: real per-sample tau_a / tau_s")
    print("=" * 70)

    check_idxs = [0, 1, len(train_ds) // 2, len(train_ds) - 1]
    taus_a, taus_s = [], []
    for idx in check_idxs:
        item = train_ds[idx]
        taus_a.append(item["tau_a"])
        taus_s.append(item["tau_s"])
        print(f"sample idx={idx:>6} | tau_a[:5]={item['tau_a'][:5].numpy()} | tau_s[:5]={item['tau_s'][:5].numpy()}")

    all_tau_a_identical = all(torch.allclose(taus_a[0], t) for t in taus_a[1:])
    all_tau_s_identical = all(torch.allclose(taus_s[0], t) for t in taus_s[1:])

    assert not all_tau_a_identical, (
        "❌ FIX 1 FAILED: tau_a is IDENTICAL across every sample checked — "
        "real timestamps are NOT reaching the model."
    )
    assert not all_tau_s_identical, (
        "❌ FIX 1 FAILED: tau_s is IDENTICAL across every sample checked — "
        "real timestamps are NOT reaching the model."
    )
    print("✅ FIX 1 VERIFIED: tau_a and tau_s vary across real samples (not synthetic/fixed).")
    print("=" * 70)

print("verify_real_taus(train_ds) ready — runs for every beta inside run_experiment.")

verify_real_taus(train_ds) ready — runs for every beta inside run_experiment.


# Cell 6: run_seed(seed) — one isolated training run of the fixed v13 configuration with the given seed (v13 seeding mechanism); keeps test-set softmax probabilities for ensembling

In [7]:
# ==============================================================================
# run_seed(seed) — ONE complete, isolated training run of the FIXED v13
# configuration with the given seed:
#   set_all_seeds(seed) -> v10 preprocessing (Ta=64, strongest-64, raw 6 IMU;
#   built once, identical for every seed) -> DataLoaders -> set_all_seeds(seed)
#   -> FRESH model (beta=1.0, mlp g(Δτ) 97 params, 4 LACA heads, Mamba 2 x
#   d_state 16 x d_conv 2) + class-weighted loss (lambda_align = lambda_cons =
#   0.1) + FRESH optimizer + scheduler -> 20-epoch training with best-val
#   checkpoint -> unseen-test evaluation (live forward pass) -> the test-set
#   softmax PROBABILITIES + true labels are kept (memory + .npy on disk) for
#   ensembling -> GPU memory freed.
#
# Seeding is exactly v13's mechanism, only the value changes: set_all_seeds(seed)
# (python random, numpy, torch CPU, every CUDA device, cuDNN deterministic) at
# run start AND right before model construction. That covers model init, the
# training DataLoader's shuffle order and worker seeds (both drawn from the
# global torch RNG) and every dropout mask. The mission split in Cell 4 keeps
# its own fixed random.seed(42), so all seeds share the same 1130 test windows.
# ==============================================================================
import gc
import hashlib

EPOCHS = 20
D_MODEL = 256
_PREP = None   # v10 preprocessing result (Ta=64), built once and reused by every condition

# Torch CPU RNG state right after model construction, printed by v10 ("[INIT]" line). v12 confirmed the winning
# 2 x (16, 2) run left the RNG in exactly this state (RNG identical to its 4-block baseline), so it must match here.
V10_RNG_FP_PREFIX = "5641b401720d"

LAMBDA_ALIGN = 0.1   # FIXED (v13: REAL, grid kept 0.1)
LAMBDA_CONS = 0.1    # FIXED (v13: REAL, grid kept 0.1)
PROBS_DIR = Path("/kaggle/working/seed_test_probs")   # per-seed test probabilities + labels (.npy), for ensembling
PROBS_DIR.mkdir(parents=True, exist_ok=True)


def _sha1_state(model, exclude):
    h = hashlib.sha1()
    for k, v in model.state_dict().items():
        if any(e in k for e in exclude):
            continue
        h.update(k.encode())
        h.update(v.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


def init_fingerprints(model):
    """full initial state (must DIFFER between seeds — proves the new seed reached model init), and the torch CPU
    RNG state right after construction (must differ too -> different shuffle order / dropout stream)."""
    return {
        "init": _sha1_state(model, ()),
        "rng": hashlib.sha1(torch.get_rng_state().numpy().tobytes()).hexdigest(),
    }


def lag_bias_param_arithmetic(g):
    """(exact learnable-parameter count, hand-checkable arithmetic) for the Linear/Tanh/Linear g(Δτ)."""
    terms = []
    for m in g.modules():
        if isinstance(m, nn.Linear):
            terms.append(f"Linear({m.in_features},{m.out_features}) weights {m.weight.numel()}")
            terms.append(f"bias {m.bias.numel()}")
    total = sum(p.numel() for p in g.parameters())
    return total, " + ".join(terms) + f" = {total} params"


def verify_lag_bias(model):
    """the mlp g(Δτ) is the original v5-v9 Linear(1,32)-Tanh-Linear(32,1), 97 params, 4 heads, beta 1.0."""
    for name in ("laca_as", "laca_sa"):
        laca = getattr(model.laca, name)
        g = laca.lag_bias
        n, arith = lag_bias_param_arithmetic(g)
        print(f"[LAG-BIAS] model.laca.{name}.lag_bias: {arith} | num_heads={laca.num_heads} | beta={laca.beta}")
        assert (isinstance(g, nn.Sequential) and len(g) == 3 and isinstance(g[1], nn.Tanh)
                and (g[0].in_features, g[0].out_features, g[2].in_features, g[2].out_features) == (1, 32, 32, 1)), (
            f"{name}.lag_bias is not the original mlp: {g}")
        assert n == 97, f"{name}.lag_bias has {n} params, expected 97"
        assert laca.num_heads == 4 and laca.beta == LACA_BETA


def mamba_param_counts(model):
    per_block = [sum(p.numel() for p in b.parameters()) for b in model.mamba_blocks]
    return sum(per_block), per_block


def verify_mamba_backbone(model):
    """Confirm, from the CONSTRUCTED blocks (not the constants), that the fixed v12-winner backbone is in use,
    and that its parameter count matches the known winning run (874,496)."""
    blocks = model.mamba_blocks
    assert len(blocks) == MAMBA_NUM_BLOCKS, f"{len(blocks)} Mamba blocks built, expected {MAMBA_NUM_BLOCKS}"
    total, per_block = mamba_param_counts(model)
    b0 = blocks[0]
    if isinstance(b0, RealMambaBlock):
        for i, b in enumerate(blocks):
            m = b.mamba
            assert (m.d_state, m.d_conv, m.expand, m.d_model) == (MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND, D_MODEL), (
                f"block {i}: d_state={m.d_state} d_conv={m.d_conv} expand={m.expand} d_model={m.d_model}")
            assert m.conv1d.kernel_size == (MAMBA_D_CONV,), f"block {i}: conv kernel {m.conv1d.kernel_size}"
            assert tuple(m.A_log.shape) == (m.d_inner, MAMBA_D_STATE), f"block {i}: A_log {tuple(m.A_log.shape)}"
        m = b0.mamba
        print(f"[MAMBA CHECK] {len(blocks)} x RealMambaBlock | d_state={m.d_state} (A_log {tuple(m.A_log.shape)}) | "
              f"d_conv={m.d_conv} (conv1d kernel {m.conv1d.kernel_size[0]}) | expand={m.expand} | d_inner={m.d_inner} | "
              f"local-conv path: {mamba_conv_path()}")
        print(f"[MAMBA PARAMS] fusion backbone total = {total:,} ({len(per_block)} blocks x {per_block[0]:,}) | "
              f"known winning v12 run = {EXPECTED_MAMBA_PARAMS:,} -> "
              f"{'MATCH ✅' if total == EXPECTED_MAMBA_PARAMS else 'MISMATCH ❌'} | "
              f"whole model = {sum(p.numel() for p in model.parameters()):,}")
        assert total == EXPECTED_MAMBA_PARAMS, (
            f"Mamba backbone has {total:,} params, the v12 winner had {EXPECTED_MAMBA_PARAMS:,}")
    else:
        print(f"[MAMBA CHECK] 🚨 {len(blocks)} x VectorizedSSMBlock (FALLBACK — no SSM state; d_state does not apply) | "
              f"d_conv={b0.conv1d.kernel_size[0]} | {total:,} params — NOT the real Mamba backbone, so the "
              f"{EXPECTED_MAMBA_PARAMS:,}-param check and the Part 0 reproduction CANNOT pass on this machine.")
    return total, per_block[0]


def verify_shapes(model, batch_t, when):
    """Real batch through the model's OWN modules; assert every shape the fusion step relies on.
    Eval mode + no_grad: no RNG consumed, no BatchNorm stats updated."""
    xa, xs, ta, ts = batch_t
    was_training = model.training
    model.eval()
    with torch.no_grad():
        za = model.audio_encoder(xa)
        zs = model.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)
        Ha = model.htt(za, ta, modality_idx=0)
        Hs = model.htt(zs, ts, modality_idx=1)
        B, Ta_, Ts_ = Ha.shape[0], Ha.shape[1], Hs.shape[1]
        H_tilde_a, H_tilde_s, aux = model.laca(Ha, Hs, ta, ts)
        assert H_tilde_a.shape == (B, Ta_, D_MODEL) and H_tilde_s.shape == (B, Ts_, D_MODEL)
        F_seq = torch.cat([H_tilde_a, model.sep_token.expand(B, -1, -1), H_tilde_s], dim=1)
        L = Ta_ + 1 + Ts_
        assert F_seq.shape == (B, L, D_MODEL), f"Mamba input {tuple(F_seq.shape)}"
        blk0_in = F_seq
        for i, blk in enumerate(model.mamba_blocks):
            F_seq = blk(F_seq)
            assert F_seq.shape == (B, L, D_MODEL) and torch.isfinite(F_seq).all(), (
                f"Mamba block {i} output {tuple(F_seq.shape)} (finite={bool(torch.isfinite(F_seq).all())})")
        logits, pooled, aux_preds, aux2 = model(xa, xs, ta, ts)
        assert logits.shape == (B, model.n_classes) and torch.isfinite(logits).all(), f"logits {tuple(logits.shape)}"
        assert pooled["z_a"].shape == pooled["z_s"].shape == (B, D_MODEL)
        assert aux2["lag_as"].shape == aux2["lag_sa"].shape == (B,)
        print(f"[SHAPES {when}] Mamba input {tuple(blk0_in.shape)} -> {len(model.mamba_blocks)} blocks, each "
              f"{(B, L, D_MODEL)} ✅ -> logits {tuple(logits.shape)} ✅")
    model.train(was_training)


def verify_loss_weights(criterion, lambda_align, lambda_cons, model, batch_t, labels_t):
    """Confirm the loss weights from the CONSTRUCTED loss module, not from the config variables:
      1. read criterion.lambda_align / criterion.lambda_cons and compare with the requested values;
      2. run the module on a real batch and check that the total it returns equals
         loss_cls + criterion.lambda_align * loss_align + criterion.lambda_cons * loss_cons,
         i.e. the stored weights are the ones actually applied inside forward().
    Eval mode + no_grad: no RNG consumed, no BatchNorm stats updated, no gradients."""
    la, lc = criterion.lambda_align, criterion.lambda_cons
    print(f"[LOSS WEIGHTS] requested: lambda_align={lambda_align} | lambda_cons={lambda_cons}")
    print(f"[LOSS WEIGHTS] read from the constructed {type(criterion).__name__}: "
          f"criterion.lambda_align={la!r} | criterion.lambda_cons={lc!r}")
    assert float(la) == float(lambda_align) and float(lc) == float(lambda_cons), (
        f"loss module holds ({la}, {lc}) but ({lambda_align}, {lambda_cons}) was requested")
    was_training = model.training
    model.eval()
    with torch.no_grad():
        logits, pooled, aux_preds, _ = model(*batch_t)
        total, comp = criterion(logits, labels_t, pooled, aux_preds)
    model.train(was_training)
    total = float(total.item())
    recon = comp["loss_cls"] + la * comp["loss_align"] + lc * comp["loss_cons"]
    naive = comp["loss_cls"] + 0.1 * comp["loss_align"] + 0.1 * comp["loss_cons"]
    print(f"[LOSS WEIGHTS] functional check on a real batch: total={total:.6f} | loss_cls={comp['loss_cls']:.6f} | "
          f"loss_align={comp['loss_align']:.6f} (x{la} -> {la * comp['loss_align']:.6f}) | "
          f"loss_cons={comp['loss_cons']:.6f} (x{lc} -> {lc * comp['loss_cons']:.6f})")
    assert all(math.isfinite(v) for v in comp.values()), f"non-finite loss component: {comp}"
    assert abs(total - recon) <= 1e-4 * max(1.0, abs(recon)), (
        f"total {total:.6f} != cls + {la}*align + {lc}*cons = {recon:.6f} — the module is NOT applying these weights")
    if (la, lc) != (0.1, 0.1):
        # the default-weight total must NOT be what forward() returned (otherwise the weights were ignored)
        assert abs(total - naive) > 1e-6, "total equals the 0.1/0.1 total — the new weights were not applied"
    for name, w in (("loss_align", la), ("loss_cons", lc)):
        if w == 0.0:
            print(f"[LOSS WEIGHTS]   {name}: weight 0.0 -> still COMPUTED (raw {comp[name]:.6f}, finite) but adds "
                  f"0.0 to the total")
    print(f"[LOSS WEIGHTS] ✅ constructed loss module applies lambda_align={la}, lambda_cons={lc} "
          f"(total == cls + {la}*align + {lc}*cons to within 1e-4)")


def run_seed(seed, tag=None, epochs=EPOCHS):
    lambda_align, lambda_cons = LAMBDA_ALIGN, LAMBDA_CONS
    global _PREP
    t_run0 = time.time()
    tag = tag or f"seed={seed}"
    set_all_seeds(seed)
    print(f"[SEEDS] reset at condition start -> random / numpy / torch / cuda = {seed}")
    print(f"[SEED RUN] {tag} | seed={seed} | fixed: lambda_align={lambda_align}, lambda_cons={lambda_cons}, beta={LACA_BETA}, "
          f"mlp g(Δτ), 4 heads, Mamba {MAMBA_NUM_BLOCKS} x (d_state {MAMBA_D_STATE}, d_conv {MAMBA_D_CONV})")

    # --- 1. v10 preprocessing (Ta=64) — identical for every condition, built on first use ---
    if _PREP is None:
        _PREP = preprocess_for_ta(TA_AUDIO)
    else:
        print(f"[DATA] reusing the Ta={TA_AUDIO} windows built for the first condition (identical data for every condition)")
    prep = _PREP

    # --- 2. Datasets / loaders (v10 settings) ---
    train_ds, val_ds, test_ds, train_loader, val_loader, test_loader = build_dataloaders(prep["paths"])
    verify_real_taus(train_ds)

    # Batch built straight from the dataset (no DataLoader iteration -> no RNG consumed).
    probe = [train_ds[i] for i in range(min(batch_size, len(train_ds)))]
    probe_t = tuple(torch.stack([p[k] for p in probe]).to(device) for k in ("x_a", "x_s", "tau_a", "tau_s"))
    probe_labels = torch.stack([p["label"] for p in probe]).to(device)
    assert tuple(probe_t[0].shape) == (len(probe), TA_AUDIO, N_AUDIO_BINS)
    assert tuple(probe_t[1].shape) == (len(probe), TS_SENSOR, 6)

    # --- 3. FRESH model (THIS seed reset right before, exactly as v5-v13) ---
    set_all_seeds(seed)
    model = BiModalAeroSyncMamba(
        d_model=D_MODEL,
        n_classes=5,
        dropout=0.15,
        temporal_bias_scale=LACA_BETA,        # FIXED at 1.0
        audio_in_dim=N_AUDIO_BINS,
        sensor_in_dim=6,
    ).to(device)
    fps = init_fingerprints(model)

    print("\n" + "-" * 90)
    verify_lag_bias(model)
    mamba_total, mamba_per_block = verify_mamba_backbone(model)
    rng_matches_v12 = fps["rng"].startswith(V10_RNG_FP_PREFIX)
    if seed == 42:
        print(f"[INIT] torch RNG after construction {fps['rng'][:12]} vs the known seed-42 baseline ({V10_RNG_FP_PREFIX}): "
              + ("IDENTICAL ✅ (same init, shuffle order + dropout masks as v12/v13)" if rng_matches_v12 else
                 "DIFFERENT ⚠️  (RNG stream differs from v12/v13; a different torch build can also cause this)"))
    ref = _SEED42_FPS if "_SEED42_FPS" in globals() else None
    if seed != 42 and ref is not None:
        diff_init, diff_rng = fps["init"] != ref["init"], fps["rng"] != ref["rng"]
        print(f"[INIT] seed {seed} vs seed 42: initial weights DIFFERENT: {'YES ✅' if diff_init else 'NO ❌'} | "
              f"post-construction RNG (-> shuffle order / dropout masks) DIFFERENT: {'YES ✅' if diff_rng else 'NO ❌'}")
        assert diff_init and diff_rng, (f"seed {seed} produced the SAME init / RNG stream as seed 42 — the new seed "
                                        f"did not reach model construction")
    verify_shapes(model, probe_t, "before training")

    _fusion_cls = type(model.mamba_blocks[0])
    print(f"[MODEL] FUSION BACKBONE IN USE: {_fusion_cls.__module__}.{_fusion_cls.__name__}"
          + ("  🚨 pure-PyTorch FALLBACK, NOT real mamba_ssm 🚨" if _fusion_cls.__name__ == "VectorizedSSMBlock"
             else "  ✅ REAL mamba_ssm selective-scan SSM"))

    # FIX 4: inverse-frequency class weights from REAL training-set class counts
    train_labels_arr = np.array([s["label"] for s in train_ds.samples])
    class_counts = np.bincount(train_labels_arr, minlength=5)
    class_weights_np = class_counts.sum() / (len(class_counts) * np.maximum(class_counts, 1))
    class_weights = torch.tensor(class_weights_np, dtype=torch.float32).to(device)
    print("[FIX 4] class weights (from real train counts): "
          + " | ".join(f"C{c}: n={class_counts[c]} w={class_weights_np[c]:.4f}" for c in range(5)))

    # --- the ONLY thing that changes between conditions: the two auxiliary loss weights ---
    criterion = BiModalAeroSyncLoss(lambda_align=lambda_align, lambda_cons=lambda_cons, class_weights=class_weights)
    verify_loss_weights(criterion, lambda_align, lambda_cons, model, probe_t, probe_labels)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    print("-" * 90 + "\n")

    best_val_f1 = 0.0
    best_weights = None
    best_epoch = None
    history = {"train_loss": [], "val_loss": [], "val_f1": [], "val_acc": [],
               "train_loss_cls": [], "train_loss_align": [], "train_loss_cons": [],
               "val_loss_cls": [], "val_loss_align": [], "val_loss_cons": []}
    ckpt_path = CACHE_DIR / f"aerosync_bimodal_best_seed{seed}.pt"

    try:
        # --- 4. Training (identical loop to v5-v12; the extra component sums only read .item() values) ---
        start_time = time.time()
        print(f"🚀 Training Bi-Modal AeroSync-Mamba ({tag}) on {len(train_ds)} real flight windows...\n")

        for epoch in range(1, epochs + 1):
            model.train()
            running_loss, running_cls = 0.0, 0.0
            running_align, running_cons = 0.0, 0.0

            pbar = tqdm(train_loader, desc=f"{tag} Epoch {epoch:02d}/{epochs}", leave=False)
            for batch in pbar:
                xa = batch["x_a"].to(device, non_blocking=True)
                xs = batch["x_s"].to(device, non_blocking=True)
                ta = batch["tau_a"].to(device, non_blocking=True)
                ts = batch["tau_s"].to(device, non_blocking=True)
                labels = batch["label"].to(device, non_blocking=True)

                optimizer.zero_grad()
                logits, pooled_reps, aux_preds, _ = model(xa, xs, ta, ts)
                loss, l_dict = criterion(logits, labels, pooled_reps, aux_preds)
                loss.backward()

                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                running_loss += loss.item()
                running_cls += l_dict["loss_cls"]
                running_align += l_dict["loss_align"]
                running_cons += l_dict["loss_cons"]
                pbar.set_postfix({"loss": f"{loss.item():.3f}", "cls": f"{l_dict['loss_cls']:.3f}"})

            scheduler.step()
            n_tr = len(train_loader)
            avg_train_loss = running_loss / n_tr

            # --- Validation ---
            model.eval()
            val_preds, val_targets, val_probs = [], [], []
            val_loss_total = 0.0
            val_comp = {"loss_cls": 0.0, "loss_align": 0.0, "loss_cons": 0.0}

            with torch.no_grad():
                for batch in val_loader:
                    xa = batch["x_a"].to(device)
                    xs = batch["x_s"].to(device)
                    ta = batch["tau_a"].to(device)
                    ts = batch["tau_s"].to(device)
                    labels = batch["label"].to(device)

                    logits, pooled_reps, aux_preds, _ = model(xa, xs, ta, ts)
                    v_loss, v_dict = criterion(logits, labels, pooled_reps, aux_preds)
                    val_loss_total += v_loss.item()
                    for k in val_comp:
                        val_comp[k] += v_dict[k]

                    probs = F.softmax(logits, dim=-1)
                    val_preds.extend(probs.argmax(dim=-1).cpu().numpy())
                    val_targets.extend(labels.cpu().numpy())
                    val_probs.extend(probs.cpu().numpy())

            avg_val_loss = val_loss_total / len(val_loader)
            val_metrics = compute_all_metrics(np.array(val_targets), np.array(val_preds), np.array(val_probs))

            history["train_loss"].append(avg_train_loss)
            history["val_loss"].append(avg_val_loss)
            history["val_f1"].append(val_metrics["macro_f1"])
            history["val_acc"].append(val_metrics["accuracy"])
            history["train_loss_cls"].append(running_cls / n_tr)
            history["train_loss_align"].append(running_align / n_tr)
            history["train_loss_cons"].append(running_cons / n_tr)
            for k in val_comp:
                history[f"val_{k}"].append(val_comp[k] / len(val_loader))

            is_best = ""
            if val_metrics["macro_f1"] > best_val_f1:
                best_val_f1 = val_metrics["macro_f1"]
                best_weights = copy.deepcopy(model.state_dict())
                best_epoch = epoch
                torch.save(best_weights, ckpt_path)
                is_best = " ⭐ [Best Saved]"

            print(
                f"{tag} Epoch {epoch:02d}/{epochs} | "
                f"Train Loss: {avg_train_loss:.4f} (cls {running_cls / n_tr:.4f} | align {running_align / n_tr:.4f} | "
                f"cons {running_cons / n_tr:.4f}) | "
                f"Val Loss: {avg_val_loss:.4f} | "
                f"Val Acc: {val_metrics['accuracy']:.4f} | "
                f"Val Macro-F1: {val_metrics['macro_f1']:.4f}{is_best}"
            )

        train_seconds = time.time() - start_time
        print(f"\n🎉 {tag} Training Finished in {train_seconds:.1f}s | Best Val Macro-F1: {best_val_f1:.4f} "
              f"(epoch {best_epoch})")

        # --- Final-epoch loss components (RAW values, before weighting) ---
        fe = {k: history[k][-1] for k in history}
        print(f"[LOSS COMPONENTS @ final epoch {epochs}] (lambda_align={criterion.lambda_align}, "
              f"lambda_cons={criterion.lambda_cons})")
        for split in ("train", "val"):
            print(f"   {split:<5}: loss_cls {fe[f'{split}_loss_cls']:.4f} | loss_align {fe[f'{split}_loss_align']:.4f} "
                  f"(weighted {criterion.lambda_align * fe[f'{split}_loss_align']:.4f}) | loss_cons "
                  f"{fe[f'{split}_loss_cons']:.4f} (weighted {criterion.lambda_cons * fe[f'{split}_loss_cons']:.4f}) | "
                  f"total {fe[f'{split}_loss']:.4f}")
        for name, w in (("loss_align", criterion.lambda_align), ("loss_cons", criterion.lambda_cons)):
            if w == 0.0:
                print(f"   ↪ {name} has weight 0.0: its raw value above is still computed every step but contributes "
                      f"nothing to the total/gradient (epoch-1 train value {history['train_' + name][0]:.4f} -> "
                      f"final {fe['train_' + name]:.4f}).")

        # --- 5. Unseen test evaluation with the best-val weights — LIVE forward pass (as every prior experiment) ---
        if best_weights is not None:
            model.load_state_dict(best_weights)
        model.eval()
        verify_shapes(model, probe_t, "best checkpoint")

        test_preds, test_targets, test_probs, lags_as_list, lags_sa_list = [], [], [], [], []
        n_test_batches = 0
        order_hash = hashlib.sha1()   # fingerprint of the test INPUTS in the order they were scored
        with torch.no_grad():
            for batch in test_loader:
                xa = batch["x_a"].to(device)
                xs = batch["x_s"].to(device)
                ta = batch["tau_a"].to(device)
                ts = batch["tau_s"].to(device)
                labels = batch["label"].to(device)

                logits, _, _, aux = model(xa, xs, ta, ts)
                probs = F.softmax(logits, dim=-1)
                test_preds.extend(probs.argmax(dim=-1).cpu().numpy())
                test_targets.extend(labels.cpu().numpy())
                test_probs.extend(probs.cpu().numpy())
                lags_as_list.extend(aux["lag_as"].cpu().numpy())
                lags_sa_list.extend(aux["lag_sa"].cpu().numpy())
                n_test_batches += 1
                order_hash.update(batch["x_a"].numpy().tobytes())
                order_hash.update(batch["tau_a"].numpy().tobytes())
                order_hash.update(batch["label"].numpy().tobytes())

        n_test = len(test_targets)
        # provenance: every test window went through THIS model in THIS call, scored by the same
        # compute_all_metrics (Cell 3) used by every prior experiment
        assert n_test_batches == len(test_loader) and n_test == len(test_ds) == len(test_preds) == len(test_probs), (
            f"test pass incomplete: {n_test_batches}/{len(test_loader)} batches, {n_test}/{len(test_ds)} windows")
        test_metrics = compute_all_metrics(np.array(test_targets), np.array(test_preds), np.array(test_probs))
        n_correct = int((np.array(test_targets) == np.array(test_preds)).sum())
        # --- keep this seed's raw test-set softmax probabilities (NOT just argmax) for ensembling ---
        probs_arr = np.asarray(test_probs, dtype=np.float64)           # (1130, 5), rows = test windows in order
        targets_arr = np.asarray(test_targets, dtype=np.int64)
        preds_arr = np.asarray(test_preds, dtype=np.int64)
        assert probs_arr.shape == (n_test, 5) and np.allclose(probs_arr.sum(axis=1), 1.0, atol=1e-4)
        assert np.array_equal(probs_arr.argmax(axis=1), preds_arr), "saved probabilities do not reproduce predictions"
        probs_path = PROBS_DIR / f"seed{seed}_test_probs.npy"
        labels_path = PROBS_DIR / f"seed{seed}_test_labels.npy"
        np.save(probs_path, probs_arr)
        np.save(labels_path, targets_arr)
        test_order_sha1 = order_hash.hexdigest()
        print(f"[PROBS] seed {seed}: test probabilities {probs_arr.shape} + labels saved -> {probs_path.name}, "
              f"{labels_path.name} | test-input order fingerprint {test_order_sha1[:12]}")
        lag_as_mean = float(np.mean(lags_as_list))
        lag_as_std = float(np.std(lags_as_list))
        lag_sa_mean = float(np.mean(lags_sa_list))
        print("=" * 65)
        print(f"     BI-MODAL AEROSYNC-MAMBA ({tag}): 100% REAL TEST BENCHMARK")
        print("=" * 65)
        print(f"  • Test Accuracy       : {test_metrics['accuracy']:.4f} ({test_metrics['accuracy']*100:.2f}%)")
        print(f"  • Macro-F1 Score      : {test_metrics['macro_f1']:.4f}")
        print(f"  • Balanced Accuracy   : {test_metrics['balanced_accuracy']:.4f}")
        print(f"  • One-vs-Rest AUROC   : {test_metrics['macro_auroc']:.4f}")
        print(f"  • Learned Audio-IMU Lag (Δ_as) : {lag_as_mean:+.4f} (normalized units) "
              f"≈ {lag_as_mean * WINDOW_SEC * 1000:+.1f} ms | std over test windows {lag_as_std:.4f}")
        print(f"  • Learned IMU-Audio Lag (Δ_sa) : {lag_sa_mean:+.4f} (normalized units)")
        print(f"  • Seed                : {seed} | loss weights lambda_align={criterion.lambda_align}, "
              f"lambda_cons={criterion.lambda_cons}")
        print(f"  • Metrics source      : LIVE forward pass, best-val checkpoint (epoch {best_epoch}), "
              f"{n_test} test windows / {n_test_batches} batches")
        print("=" * 65)
    finally:
        # --- 6. Free GPU memory so sequential conditions never accumulate ---
        del model, optimizer, scheduler, criterion, best_weights, probe_t, probe, probe_labels
        del train_loader, val_loader, test_loader, train_ds, val_ds, test_ds
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            print(f"[GPU] freed | allocated={torch.cuda.memory_allocated() / 1024**2:.1f} MB "
                  f"| reserved={torch.cuda.memory_reserved() / 1024**2:.1f} MB")

    return {
        "seed": seed,
        "lambda_align": float(lambda_align),
        "lambda_cons": float(lambda_cons),
        "accuracy": float(test_metrics["accuracy"]),
        "macro_f1": float(test_metrics["macro_f1"]),
        "balanced_accuracy": float(test_metrics["balanced_accuracy"]),
        "macro_auroc": float(test_metrics["macro_auroc"]),
        "n_test_correct": n_correct,
        "final_train_loss": round(fe["train_loss"], 6),
        "final_train_loss_cls": round(fe["train_loss_cls"], 6),
        "final_train_loss_align": round(fe["train_loss_align"], 6),
        "final_train_loss_cons": round(fe["train_loss_cons"], 6),
        "final_val_loss_cls": round(fe["val_loss_cls"], 6),
        "final_val_loss_align": round(fe["val_loss_align"], 6),
        "final_val_loss_cons": round(fe["val_loss_cons"], 6),
        "learned_lag_as_mean": round(lag_as_mean, 6),
        "learned_lag_as_std": round(lag_as_std, 6),
        "learned_lag_as_ms": round(lag_as_mean * WINDOW_SEC * 1000, 2),
        "learned_lag_sa_mean": round(lag_sa_mean, 6),
        "best_val_macro_f1": round(float(best_val_f1), 6),
        "best_epoch": best_epoch,
        "mamba_params": mamba_total,
        "conv_path": mamba_conv_path(),
        "rng_matches_v12": rng_matches_v12,
        "init_sha1": fps["init"][:12],
        "rng_sha1": fps["rng"][:12],
        "test_order_sha1": test_order_sha1,
        "probs_path": str(probs_path),
        "n_test_windows": n_test,
        "metrics_source": f"live forward pass (best-val checkpoint epoch {best_epoch}, {n_test} test windows)",
        "train_seconds": round(train_seconds, 1),
        "run_seconds": round(time.time() - t_run0, 1),
        # reproduction fingerprints (not written to the CSV)
        "_fps": fps,
        "_probs": probs_arr,
        "_targets": targets_arr,
        "_n_windows": prep["n_windows"],
        "_class_counts": class_counts.tolist(),
        "_sensor_fingerprint": prep["sensor_fingerprint"],
        "_epoch1": {"train_loss": history["train_loss"][0], "val_loss": history["val_loss"][0],
                    "val_acc": history["val_acc"][0], "val_f1": history["val_f1"][0]},
    }

WINDOW_SEC = 2.56   # slice_real_bimodal window length; tau is normalized by it in RealBiModalDataset
print(f"run_seed(seed) ready | lambda_align={LAMBDA_ALIGN}, lambda_cons={LAMBDA_CONS} fixed | fixed Mamba {MAMBA_NUM_BLOCKS} x (d_state {MAMBA_D_STATE}, "
      f"d_conv {MAMBA_D_CONV}), conv path: {mamba_conv_path()} | fixed: beta={LACA_BETA}, mlp g(Δτ) 97 params, "
      f"4 LACA heads, Ta={TA_AUDIO}, N={N_AUDIO_BINS}, Ts={TS_SENSOR}")


run_seed(seed) ready | lambda_align=0.1, lambda_cons=0.1 fixed | fixed Mamba 2 x (d_state 16, d_conv 2), conv path: fused (causal-conv1d CUDA kernel) | fixed: beta=1.0, mlp g(Δτ) 97 params, 4 LACA heads, Ta=64, N=64, Ts=128


# Cell 7: PART 0 — seed = 42 must reproduce the known baseline (acc 0.9080, macro-F1 0.9048, bal-acc 0.9022, AUROC 0.9894) before the other seeds

In [8]:
# ==============================================================================
# PART 0 — seed = 42 must reproduce the known baseline BEFORE the other 4 seeds.
# The seed-42 run is trained ONCE here and serves as both the reproduction check
# and the seed-42 ensemble member. It cannot be skipped by reusing v13's numbers:
# ensembling needs its per-window test PROBABILITIES, and no earlier version
# saved those.
# Results CSV (seed rows now; ensemble rows appended in Part 2) lives here too.
# ==============================================================================
import csv
import traceback
from datetime import datetime

SEEDS = [42, 123, 2024, 7, 99]   # training order; ensembles use the FIRST k of these
RESULTS_CSV = Path("/kaggle/working/multi_seed_ensemble_results.csv")
RESULTS_CSV_FIELDS = ["row_type", "name", "seed", "n_members", "members",
                      "accuracy", "macro_f1", "balanced_accuracy", "macro_auroc", "n_test_correct", "n_test_windows",
                      "best_val_macro_f1", "best_epoch",
                      "final_train_loss", "final_train_loss_cls", "final_train_loss_align", "final_train_loss_cons",
                      "learned_lag_as_mean", "learned_lag_as_ms", "learned_lag_sa_mean",
                      "mamba_params", "conv_path", "init_sha1", "rng_sha1", "rng_matches_v12", "test_order_sha1",
                      "probs_path", "metrics_source", "status", "train_seconds", "run_seconds", "note", "finished_at"]

# If True, the other 4 seeds do NOT run when seed = 42 fails to reproduce the known baseline.
REQUIRE_BASELINE_REPRODUCTION = True

# KNOWN BASELINE (seed = 42) — v12 Stage C winner, reproduced bit-for-bit by v13 Part 0.
KNOWN_BASELINE = {"accuracy": 0.9080, "macro_f1": 0.9048, "balanced_accuracy": 0.9022, "macro_auroc": 0.9894}
KNOWN_LAG_AS = -0.0291
KNOWN_BEST_EPOCH = 17
KNOWN_EPOCH1 = {"train_loss": 1.4173, "val_loss": 0.8880, "val_acc": 0.8197, "val_f1": 0.8280}
# Deterministic fingerprints of the v7-v13 data path — independent of GPU numerics, must match EXACTLY.
V10_BIN_INDICES_N64 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255,
                       258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508,
                       509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760,
                       761, 762, 763, 764, 765, 766]
V10_N_WINDOWS = (4934, 954, 1130)
V10_CLASS_COUNTS = [906, 1020, 1004, 1008, 996]
V10_SENSOR_FINGERPRINT = {
    "TRAIN": (4934, -6167105.225, 808427.456, (906, 1020, 1004, 1008, 996)),
    "VAL":   (954, -1193206.652, 156301.664, (126, 229, 199, 190, 210)),
    "TEST":  (1130, -1413563.944, 185127.438, (286, 205, 210, 228, 201)),
}
REPRO_TOL = 0.0005

SEED_RESULTS = {}   # seed -> full result dict incl. "_probs" (1130 x 5) and "_targets" (1130,), kept for ensembling


def append_row(row):
    out = {k: row.get(k) for k in RESULTS_CSV_FIELDS}
    out["finished_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    write_header = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=RESULTS_CSV_FIELDS)
        if write_header:
            w.writeheader()
        w.writerow(out)
    print(f"💾 appended {row['row_type']} / {row['name']} to {RESULTS_CSV} ({'header + row' if write_header else 'row'})")


def assert_metrics_provenance(row):
    """Seed rows: live forward pass over the full test set (as every prior experiment). Ensemble rows: computed
    from those seeds' saved test probabilities on the same 1130 windows."""
    if row["row_type"] == "seed":
        assert str(row["metrics_source"]).startswith("live forward pass"), (
            f"seed {row['seed']}: metrics not from a live forward pass")
    else:
        assert str(row["metrics_source"]).startswith("ensemble of saved test probabilities"), (
            f"{row['name']}: ensemble row with an unexpected metrics source")
    assert int(row["n_test_windows"]) == V10_N_WINDOWS[2], (
        f"{row['name']}: scored on {row['n_test_windows']} test windows, expected {V10_N_WINDOWS[2]}")
    return True


def print_seed_metrics(res):
    """Immediately after a seed finishes: its standalone test metrics."""
    print("\n" + "*" * 90)
    print(f"*  SEED {res['seed']} FINISHED — standalone test metrics (live forward pass, {res['n_test_windows']} windows)")
    print(f"*    accuracy {res['accuracy']:.4f} | macro-F1 {res['macro_f1']:.4f} | balanced accuracy "
          f"{res['balanced_accuracy']:.4f} | macro-AUROC {res['macro_auroc']:.4f} | best epoch {res['best_epoch']} "
          f"| train {res['train_seconds']:.0f}s")
    print("*" * 90)


def train_and_log_seed(seed, note=""):
    """Train one seed, keep its probabilities, print its metrics right away, append its CSV row."""
    res = run_seed(seed, tag=f"seed={seed}")
    res.update({"row_type": "seed", "name": f"seed_{seed}", "n_members": 1, "members": str(seed),
                "status": "computed", "note": note})
    assert_metrics_provenance(res)
    SEED_RESULTS[seed] = res
    print_seed_metrics(res)
    return res


def check_seed42_against_known(res):
    """Loud reproduction check for seed 42. Returns (ok, one-line verdict)."""
    print("\n" + "=" * 90)
    print("  PART 0 CHECK — does seed = 42 reproduce the known baseline?")
    print("=" * 90)
    det = {
        "strongest-64 bin indices": SELECTED_BIN_INDICES.tolist() == V10_BIN_INDICES_N64,
        f"train/val/test windows {V10_N_WINDOWS}": tuple(res["_n_windows"]) == V10_N_WINDOWS,
        f"train class counts {V10_CLASS_COUNTS}": res["_class_counts"] == V10_CLASS_COUNTS,
        "raw sensor values/timestamps/labels (v10 fingerprint)": res["_sensor_fingerprint"] == V10_SENSOR_FINGERPRINT,
        f"Mamba backbone params = {EXPECTED_MAMBA_PARAMS:,}": int(res["mamba_params"]) == EXPECTED_MAMBA_PARAMS,
    }
    for name, ok in det.items():
        print(f"  [deterministic] {name:<53}: {'MATCH ✅' if ok else 'MISMATCH ❌'}")
    print(f"  [init]          {'torch RNG after construction vs v12/v13 (' + V10_RNG_FP_PREFIX + ')':<53}: "
          f"{'MATCH ✅' if res['rng_matches_v12'] else 'MISMATCH ⚠️'}")
    e1 = res["_epoch1"]
    print("  [epoch 1]       " + " | ".join(
        f"{k}: known {KNOWN_EPOCH1[k]:.4f} vs now {e1[k]:.4f} ({e1[k] - KNOWN_EPOCH1[k]:+.4f})" for k in KNOWN_EPOCH1))

    print(f"\n  {'metric':<20} {'known':>10} {'this run':>10} {'delta':>10}")
    worst = 0.0
    for k, v in KNOWN_BASELINE.items():
        d = res[k] - v
        worst = max(worst, abs(d))
        print(f"  {k:<20} {v:>10.4f} {res[k]:>10.4f} {d:>+10.4f}{'' if abs(d) <= REPRO_TOL else '   <-- differs'}")
    print(f"  {'learned lag Δ_as':<20} {KNOWN_LAG_AS:>+10.4f} {res['learned_lag_as_mean']:>+10.4f} "
          f"{res['learned_lag_as_mean'] - KNOWN_LAG_AS:>+10.4f}   (v12/v13 value; informational)")
    print(f"  {'best epoch':<20} {KNOWN_BEST_EPOCH:>10} {res['best_epoch']:>10}   (informational)")

    if not all(det.values()):
        ok, verdict = False, "PORT BUG: deterministic data / backbone fingerprint mismatch"
        print("\n" + "🚨" * 40)
        print("🚨  PART 0 FAILED — bins / windows / class counts / sensor values / Mamba param count differ from v13,")
        print("🚨  which cannot be GPU noise: the data path or the backbone changed in the v13 -> v15 port.")
        print("🚨" * 40)
    elif worst <= REPRO_TOL:
        ok, verdict = True, "reproduces known baseline (all 4 metrics within 0.0005)"
        print(f"\n✅ PART 0 PASSED: seed = 42 REPRODUCES the known baseline — all four test metrics within ±{REPRO_TOL} "
              f"and all deterministic fingerprints match.")
    else:
        ok, verdict = False, f"DOES NOT reproduce known metrics (max |delta| {worst:.4f}); data fingerprints match"
        print("\n" + "🚨" * 40)
        print(f"🚨  PART 0 FAILED — SEED-42 TEST METRICS DIFFER FROM THE KNOWN BASELINE (largest |delta| = {worst:.4f}).")
        print("🚨  Data path is identical, so the difference arose in training. Check the [init] and epoch-1 lines:")
        print("🚨  if the RNG or epoch 1 already differ, the model/RNG path changed in the port; if they match and")
        print("🚨  later epochs drift, it is GPU non-determinism.")
        print("🚨" * 40)
    print("=" * 90)
    return ok, verdict


if RESULTS_CSV.exists():
    _backup = RESULTS_CSV.with_suffix(f".{datetime.now().strftime('%Y%m%d_%H%M%S')}.bak.csv")
    RESULTS_CSV.rename(_backup)
    print(f"ℹ️  existing results file moved to {_backup}")

_sweep_t0 = time.time()
BASELINE_ROW, BASELINE_OK, BASELINE_VERDICT = None, False, "not run"
print("\n\n" + "=" * 90)
print(f"=== PART 0: seed = 42 (reference) | lambda_align={LAMBDA_ALIGN}, lambda_cons={LAMBDA_CONS} | Mamba "
      f"{MAMBA_NUM_BLOCKS} x (d_state {MAMBA_D_STATE}, d_conv {MAMBA_D_CONV}) ===")
print("=" * 90)
try:
    BASELINE_ROW = train_and_log_seed(42, note="reference seed")
    _SEED42_FPS = BASELINE_ROW["_fps"]
    BASELINE_OK, BASELINE_VERDICT = check_seed42_against_known(BASELINE_ROW)
    BASELINE_ROW["note"] = f"reference seed; {BASELINE_VERDICT}"
    append_row(BASELINE_ROW)
except Exception as exc:
    BASELINE_VERDICT = f"seed-42 run FAILED: {exc!r}"
    print(f"\n❌ PART 0 seed-42 run FAILED: {exc!r}")
    traceback.print_exc()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

OTHER_SEEDS_ENABLED = BASELINE_OK or (BASELINE_ROW is not None and not REQUIRE_BASELINE_REPRODUCTION)
if not BASELINE_OK:
    print("\n" + "🚨" * 40)
    print(f"🚨  PART 0 did not pass ({BASELINE_VERDICT}).")
    print("🚨  " + ("The other 4 seeds will NOT run (REQUIRE_BASELINE_REPRODUCTION=True). Fix the reproduction first, "
                   "or set it to False to run them anyway." if REQUIRE_BASELINE_REPRODUCTION else
                   "REQUIRE_BASELINE_REPRODUCTION=False -> the other seeds run anyway; treat results with suspicion."))
    print("🚨" * 40)
else:
    print(f"\n✅ Part 0 verified (seed = 42 reproduces the known baseline) — proceeding to seeds {SEEDS[1:]}. "
          f"Elapsed {(time.time() - _sweep_t0) / 60:.1f} min.")




=== PART 0: seed = 42 (reference) | lambda_align=0.1, lambda_cons=0.1 | Mamba 2 x (d_state 16, d_conv 2) ===
[SEEDS] reset at condition start -> random / numpy / torch / cuda = 42
[SEED RUN] seed=42 | seed=42 | fixed: lambda_align=0.1, lambda_cons=0.1, beta=1.0, mlp g(Δτ), 4 heads, Mamba 2 x (d_state 16, d_conv 2)

--- Preprocessing TRAIN split, Ta=64 (frozen strongest-64 bins) ---


Train Class 0 (Ta=64):   0%|          | 0/13 [00:00<?, ?it/s]

[STRONGEST-64 VERIFICATION] audio chunk shape (raw waveform): (5103, 1536) | np.fft.rfft output shape: (5103, 769) | final a_feats (sqrt-compressed, frozen strongest-64 bins) shape: (5103, 64)
[STRONGEST-64 VERIFICATION] Confirmed: features were produced via np.fft.rfft (frequency-domain magnitude spectrum) using the frozen strongest-64 bin indices, NOT the first 64 bins.


Train Class 1 (Ta=64):   0%|          | 0/14 [00:00<?, ?it/s]

Train Class 2 (Ta=64):   0%|          | 0/14 [00:00<?, ?it/s]

...

seed=42 Epoch 19/20:   0%|          | 0/155 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d87e6b10540>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d87e6b10540>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

seed=42 Epoch 19/20 | Train Loss: 0.2930 (cls 0.1225 | align 1.2644 | cons 0.4403) | Val Loss: 0.8428 | Val Acc: 0.8679 | Val Macro-F1: 0.8738


seed=42 Epoch 20/20:   0%|          | 0/155 [00:00<?, ?it/s]

seed=42 Epoch 20/20 | Train Loss: 0.3018 (cls 0.1310 | align 1.2678 | cons 0.4403) | Val Loss: 0.7145 | Val Acc: 0.8910 | Val Macro-F1: 0.8951

🎉 seed=42 Training Finished in 148.7s | Best Val Macro-F1: 0.9024 (epoch 17)
[LOSS COMPONENTS @ final epoch 20] (lambda_align=0.1, lambda_cons=0.1)
   train: loss_cls 0.1310 | loss_align 1.2678 (weighted 0.1268) | loss_cons 0.4403 (weighted 0.0440) | total 0.3018
   val  : loss_cls 0.4146 | loss_align 2.5460 (weighted 0.2546) | loss_cons 0.4531 (weighted 0.0453) | total 0.7145
[SHAPES best checkpoint] Mamba input (32, 193, 256) -> 2 blocks, each (32, 193, 256) ✅ -> logits (32, 5) ✅
[PROBS] seed 42: test probabilities (1130, 5) + labels saved -> seed42_test_probs.npy, seed42_test_labels.npy | test-input order fingerprint 169980220248
     BI-MODAL AEROSYNC-MAMBA (seed=42): 100% REAL TEST BENCHMARK
  • Test Accuracy       : 0.9080 (90.80%)
  • Macro-F1 Score      : 0.9048
  • Balanced Accuracy   : 0.9022
  • One-vs-Rest AUROC   : 0.9894
  • Learn

# Cell 8: PART 1 — remaining seeds 123, 2024, 7, 99 (metrics printed + CSV row appended as each finishes)

In [9]:
# ==============================================================================
# PART 1 — the remaining 4 seeds (123, 2024, 7, 99), same configuration, same
# v13 seeding mechanism with the new seed value. Each seed's metrics print as
# soon as it finishes, and its row is appended to the CSV immediately, so
# partial results survive a session timeout. Probabilities are kept in
# SEED_RESULTS and on disk (PROBS_DIR).
# ==============================================================================
seed_failed = []

if not OTHER_SEEDS_ENABLED:
    print("⛔ PART 1 SKIPPED — seed = 42 did not reproduce the known baseline (see the banner above). Set "
          "REQUIRE_BASELINE_REPRODUCTION=False in the Part 0 cell to run the other seeds anyway.")
else:
    for i, seed in enumerate(SEEDS[1:], start=2):
        print("\n\n" + "=" * 90)
        print(f"=== SEED {i}/{len(SEEDS)}: seed = {seed} | elapsed {(time.time() - _sweep_t0) / 60:.1f} min ===")
        print("=" * 90)
        try:
            append_row(train_and_log_seed(seed))
        except Exception as exc:
            seed_failed.append((seed, repr(exc)))
            print(f"\n❌ seed {seed} FAILED: {exc!r}")
            traceback.print_exc()
            print("↪️  continuing; this seed is excluded from the statistics and ensembles.")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    print("\n" + "=" * 90)
    print(f"PART 1 COMPLETE in {(time.time() - _sweep_t0) / 60:.1f} min (incl. seed 42) | seeds finished: "
          f"{[s for s in SEEDS if s in SEED_RESULTS]} | results: {RESULTS_CSV}")
    for s, err in seed_failed:
        print(f"  ❌ seed {s}: {err}")
    print("=" * 90)




=== SEED 2/5: seed = 123 | elapsed 4.2 min ===
[SEEDS] reset at condition start -> random / numpy / torch / cuda = 123
[SEED RUN] seed=123 | seed=123 | fixed: lambda_align=0.1, lambda_cons=0.1, beta=1.0, mlp g(Δτ), 4 heads, Mamba 2 x (d_state 16, d_conv 2)
[DATA] reusing the Ta=64 windows built for the first condition (identical data for every condition)
DataLoaders ready -> Train: 4934, Val: 954, Test: 1130 windows
VERIFICATION HOOK — FIX 1: real per-sample tau_a / tau_s
sample idx=     0 | tau_a[:5]=[0.         0.01487262 0.02974525 0.04461787 0.0669268 ] | tau_s[:5]=[0.         0.00693516 0.0123625  0.02291758 0.02752617]
sample idx=     1 | tau_a[:5]=[0.00566915 0.02054177 0.03541439 0.05028701 0.06515963] | tau_s[:5]=[0.00062656 0.01503398 0.02235313 0.03039492 0.03418203]
sample idx=  2467 | tau_a[:5]=[0.00040117 0.01437355 0.03533211 0.04930449 0.06327687] | tau_s[:5]=[0.00869414 0.01280703 0.02074766 0.02835352 0.03259258]
sample idx=  4933 | tau_a[:5]=[0.00544735 0.01980871 

seed=123 Epoch 01/20:   0%|          | 0/155 [00:00<?, ?it/s]

seed=123 Epoch 01/20 | Train Loss: 1.4048 (cls 1.0464 | align 3.1728 | cons 0.4112) | Val Loss: 0.9769 | Val Acc: 0.7652 | Val Macro-F1: 0.7762 ⭐ [Best Saved]


seed=123 Epoch 02/20:   0%|          | 0/155 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d87e6b10540>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
          Exception ignored in:  ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7d87e6b10540>^
^Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    ^^self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    if w.is_alive():^
^ ^ ^ ^ ^ ^ ^^ ^^^^^^^

...

seed=99 Epoch 18/20 | Train Loss: 0.3198 (cls 0.1489 | align 1.2763 | cons 0.4330) | Val Loss: 0.9647 | Val Acc: 0.8344 | Val Macro-F1: 0.8414


seed=99 Epoch 19/20:   0%|          | 0/155 [00:00<?, ?it/s]

seed=99 Epoch 19/20 | Train Loss: 0.2892 (cls 0.1179 | align 1.2753 | cons 0.4375) | Val Loss: 0.7544 | Val Acc: 0.8784 | Val Macro-F1: 0.8822


seed=99 Epoch 20/20:   0%|          | 0/155 [00:00<?, ?it/s]

seed=99 Epoch 20/20 | Train Loss: 0.2911 (cls 0.1209 | align 1.2657 | cons 0.4364) | Val Loss: 0.7268 | Val Acc: 0.8836 | Val Macro-F1: 0.8884 ⭐ [Best Saved]

🎉 seed=99 Training Finished in 164.6s | Best Val Macro-F1: 0.8884 (epoch 20)
[LOSS COMPONENTS @ final epoch 20] (lambda_align=0.1, lambda_cons=0.1)
   train: loss_cls 0.1209 | loss_align 1.2657 (weighted 0.1266) | loss_cons 0.4364 (weighted 0.0436) | total 0.2911
   val  : loss_cls 0.4258 | loss_align 2.5553 (weighted 0.2555) | loss_cons 0.4549 (weighted 0.0455) | total 0.7268
[SHAPES best checkpoint] Mamba input (32, 193, 256) -> 2 blocks, each (32, 193, 256) ✅ -> logits (32, 5) ✅
[PROBS] seed 99: test probabilities (1130, 5) + labels saved -> seed99_test_probs.npy, seed99_test_labels.npy | test-input order fingerprint 169980220248
     BI-MODAL AEROSYNC-MAMBA (seed=99): 100% REAL TEST BENCHMARK
  • Test Accuracy       : 0.8726 (87.26%)
  • Macro-F1 Score      : 0.8675
  • Balanced Accuracy   : 0.8646
  • One-vs-Rest AUROC   : 0

# Cell 9: PART 2 — alignment check (same 1130 test windows, same order) + probability-averaged ensembles of the first 2 / 3 / 4 / 5 seeds

In [10]:
# ==============================================================================
# PART 2 — ENSEMBLES. First verify that every seed's saved probability array
# covers the SAME 1130 test windows in the SAME order; then average the
# probabilities of the first k seeds (k = 2..5, training order), take argmax,
# and score with the same compute_all_metrics as every individual seed.
# ==============================================================================
ENSEMBLE_RESULTS = {}   # k -> metrics dict
ENS_SEEDS = [s for s in SEEDS if s in SEED_RESULTS]   # finished seeds, in training order

if len(ENS_SEEDS) < 2:
    print(f"⛔ PART 2 SKIPPED — only {len(ENS_SEEDS)} seed(s) finished ({ENS_SEEDS}); an ensemble needs at least 2.")
else:
    if ENS_SEEDS != SEEDS:
        print(f"⚠️  only {len(ENS_SEEDS)}/{len(SEEDS)} seeds finished — ensembles use {ENS_SEEDS} "
              f"(missing: {[s for s in SEEDS if s not in SEED_RESULTS]})")

    # --- 1. ALIGNMENT: same 1130 windows, same order, for every seed ---
    print("=" * 100)
    print("  ALIGNMENT CHECK — are all saved probability arrays on the SAME test windows in the SAME order?")
    print("=" * 100)
    ref_seed = ENS_SEEDS[0]
    ref_targets = SEED_RESULTS[ref_seed]["_targets"]
    ref_order = SEED_RESULTS[ref_seed]["test_order_sha1"]
    for s in ENS_SEEDS:
        r = SEED_RESULTS[s]
        p, t = r["_probs"], r["_targets"]
        # the arrays on disk are the ones that were saved from this run
        disk_p, disk_t = np.load(r["probs_path"]), np.load(str(r["probs_path"]).replace("_probs.npy", "_labels.npy"))
        same_labels = np.array_equal(t, ref_targets)
        same_order = r["test_order_sha1"] == ref_order
        same_disk = np.array_equal(disk_p, p) and np.array_equal(disk_t, t)
        # the saved probabilities reproduce this seed's own logged metrics
        m = compute_all_metrics(t, p.argmax(axis=1), p)
        same_metrics = all(abs(m[k] - r[k]) < 1e-12 for k in ("accuracy", "macro_f1", "balanced_accuracy", "macro_auroc"))
        print(f"  seed {s:>4}: probs {p.shape} | {len(t)} true labels identical to seed {ref_seed}'s: "
              f"{'YES ✅' if same_labels else 'NO ❌'} | test-input order fingerprint {r['test_order_sha1'][:12]} "
              f"{'= ✅' if same_order else '≠ ❌'} | on-disk copy identical: {'YES ✅' if same_disk else 'NO ❌'} | "
              f"probs reproduce its logged metrics: {'YES ✅' if same_metrics else 'NO ❌'}")
        assert p.shape == (V10_N_WINDOWS[2], 5) and len(t) == V10_N_WINDOWS[2], f"seed {s}: wrong array size"
        assert same_labels and same_order, f"seed {s}: test windows / order differ from seed {ref_seed} — ensemble would be misaligned"
        assert same_disk and same_metrics, f"seed {s}: saved probabilities do not match this seed's own evaluation"
    print(f"✅ all {len(ENS_SEEDS)} probability arrays cover the same {V10_N_WINDOWS[2]} test windows in the same order "
          f"(identical true-label arrays + identical input-order fingerprints) — safe to average element-wise.")

    # --- 2. ENSEMBLES of the first k seeds ---
    print("\n" + "=" * 100)
    print(f"  ENSEMBLES — element-wise mean of the first k seeds' softmax probabilities, then argmax (order {ENS_SEEDS})")
    print("=" * 100)
    for k in range(2, len(ENS_SEEDS) + 1):
        members = ENS_SEEDS[:k]
        avg = np.mean([SEED_RESULTS[s]["_probs"] for s in members], axis=0)
        assert avg.shape == (V10_N_WINDOWS[2], 5) and np.allclose(avg.sum(axis=1), 1.0, atol=1e-4)
        preds = avg.argmax(axis=1)
        m = compute_all_metrics(ref_targets, preds, avg)
        n_correct = int((preds == ref_targets).sum())
        ENSEMBLE_RESULTS[k] = dict(m, members=members, n_correct=n_correct)
        print(f"  {k}-seed ensemble {members}: accuracy {m['accuracy']:.4f} | macro-F1 {m['macro_f1']:.4f} | "
              f"balanced accuracy {m['balanced_accuracy']:.4f} | macro-AUROC {m['macro_auroc']:.4f} "
              f"({n_correct}/{V10_N_WINDOWS[2]} correct)")
        append_row({"row_type": "ensemble", "name": f"ensemble_first_{k}", "seed": None, "n_members": k,
                    "members": " ".join(map(str, members)), **m, "n_test_correct": n_correct,
                    "n_test_windows": V10_N_WINDOWS[2], "test_order_sha1": ref_order,
                    "metrics_source": f"ensemble of saved test probabilities (mean of {k} seeds, argmax)",
                    "status": "computed", "note": "first-k seeds in training order"})


  ALIGNMENT CHECK — are all saved probability arrays on the SAME test windows in the SAME order?
  seed   42: probs (1130, 5) | 1130 true labels identical to seed 42's: YES ✅ | test-input order fingerprint 169980220248 = ✅ | on-disk copy identical: YES ✅ | probs reproduce its logged metrics: YES ✅
  seed  123: probs (1130, 5) | 1130 true labels identical to seed 42's: YES ✅ | test-input order fingerprint 169980220248 = ✅ | on-disk copy identical: YES ✅ | probs reproduce its logged metrics: YES ✅
  seed 2024: probs (1130, 5) | 1130 true labels identical to seed 42's: YES ✅ | test-input order fingerprint 169980220248 = ✅ | on-disk copy identical: YES ✅ | probs reproduce its logged metrics: YES ✅
  seed    7: probs (1130, 5) | 1130 true labels identical to seed 42's: YES ✅ | test-input order fingerprint 169980220248 = ✅ | on-disk copy identical: YES ✅ | probs reproduce its logged metrics: YES ✅
  seed   99: probs (1130, 5) | 1130 true labels identical to seed 42's: YES ✅ | test-input orde

# Cell 10: Final Report — per-seed metrics, measured seed-to-seed mean ± std, ensemble-size curve, 5-seed ensemble vs best single seed and vs measured std

In [11]:
# ==============================================================================
# FINAL REPORT — per-seed metrics, the MEASURED seed-to-seed mean ± std, every
# ensemble size, the 5-seed ensemble vs the BEST single seed, and the ensemble's
# gain over the mean seed judged against the measured std (not the old ±0.0005
# same-seed placeholder). Seed / ensemble rows are also reloaded from the CSV.
# ==============================================================================
METRICS = ["accuracy", "macro_f1", "balanced_accuracy", "macro_auroc"]
try:
    from IPython.display import display
except Exception:
    display = lambda t: print(t.to_string(index=False))
pd.set_option("display.width", 260)

if RESULTS_CSV.exists():
    _csv = pd.read_csv(RESULTS_CSV)
    for _, row in _csv.iterrows():
        assert_metrics_provenance(row.to_dict())
    print(f"✅ CSV provenance: {int((_csv['row_type'] == 'seed').sum())} seed row(s) from live test passes, "
          f"{int((_csv['row_type'] == 'ensemble').sum())} ensemble row(s) from saved probabilities | {RESULTS_CSV}")

done = [s for s in SEEDS if s in SEED_RESULTS]
if not done:
    print("⛔ no seed finished — nothing to report.")
else:
    # --- 1. individual seeds ---
    seed_df = pd.DataFrame([{"seed": s, **{k: SEED_RESULTS[s][k] for k in METRICS},
                             "best_epoch": SEED_RESULTS[s]["best_epoch"],
                             "train_seconds": SEED_RESULTS[s]["train_seconds"]} for s in done])
    print("=" * 100)
    print(f"  INDIVIDUAL SEEDS ({len(done)} rows, training order)")
    print("=" * 100)
    display(seed_df.round(4))

    # --- 2. measured seed-to-seed variance ---
    print("\n" + "=" * 100)
    print(f"  SEED-TO-SEED VARIANCE — mean ± sample std (ddof=1) across {len(done)} independent seeds")
    print("=" * 100)
    SEED_MEAN = {k: float(seed_df[k].mean()) for k in METRICS}
    SEED_STD = {k: float(seed_df[k].std(ddof=1)) if len(done) > 1 else float("nan") for k in METRICS}
    for k in METRICS:
        print(f"  {k + ':':<19} {SEED_MEAN[k]:.4f} +/- {SEED_STD[k]:.4f} across {len(done)} seeds "
              f"(min {seed_df[k].min():.4f}, max {seed_df[k].max():.4f}, range {seed_df[k].max() - seed_df[k].min():.4f})")
    if len(done) > 1:
        print(f"  -> the real seed-to-seed noise floor on macro-F1 is ±{SEED_STD['macro_f1']:.4f} (1 std), "
              f"{SEED_STD['macro_f1'] / 0.0005:.1f}x the old ±0.0005 same-seed reproducibility placeholder.")
        print(f"  -> seed 42 (the known baseline) sits at macro-F1 {SEED_RESULTS[42]['macro_f1']:.4f}, "
              f"{(SEED_RESULTS[42]['macro_f1'] - SEED_MEAN['macro_f1']) / SEED_STD['macro_f1']:+.2f} std from the seed mean."
              if 42 in SEED_RESULTS and SEED_STD["macro_f1"] > 0 else "")

    # --- 3. every ensemble size ---
    if ENSEMBLE_RESULTS:
        best_seed = max(done, key=lambda s: SEED_RESULTS[s]["macro_f1"])
        best = SEED_RESULTS[best_seed]
        rows = [{"n_seeds": 1, "members": str(done[0]), **{k: SEED_RESULTS[done[0]][k] for k in METRICS}}]
        rows += [{"n_seeds": k, "members": " ".join(map(str, e["members"])), **{m: e[m] for m in METRICS}}
                 for k, e in sorted(ENSEMBLE_RESULTS.items())]
        ens_df = pd.DataFrame(rows)
        ens_df["Δ macro_f1 vs previous size"] = ens_df["macro_f1"].diff()
        ens_df["Δ macro_f1 vs mean seed"] = ens_df["macro_f1"] - SEED_MEAN["macro_f1"]
        ens_df["Δ macro_f1 vs best seed"] = ens_df["macro_f1"] - best["macro_f1"]
        print("\n" + "=" * 100)
        print("  ENSEMBLE SIZE CURVE — first k seeds (k = 1 is the first seed alone, normally seed 42)")
        print("=" * 100)
        display(ens_df.round(4))
        k_max = max(ENSEMBLE_RESULTS)
        full = ENSEMBLE_RESULTS[k_max]
        if len(done) > 1 and SEED_STD["macro_f1"] > 0:
            within = [k for k in sorted(ENSEMBLE_RESULTS) if full["macro_f1"] - ENSEMBLE_RESULTS[k]["macro_f1"]
                      <= SEED_STD["macro_f1"]]
            print(f"  practical minimum: the smallest ensemble within 1 seed-std ({SEED_STD['macro_f1']:.4f}) of the "
                  f"{k_max}-seed ensemble's macro-F1 is k = {within[0]} "
                  f"(macro-F1 {ENSEMBLE_RESULTS[within[0]]['macro_f1']:.4f} vs {full['macro_f1']:.4f}).")

        # --- 4. full ensemble vs BEST single seed (the hardest bar) ---
        print("\n" + "#" * 100)
        print(f"#  {k_max}-SEED ENSEMBLE vs the BEST INDIVIDUAL SEED (seed {best_seed}, chosen by macro-F1)")
        print("#" * 100)
        for m in METRICS:
            print(f"#    {m:<18} ensemble {full[m]:.4f} | best seed {best[m]:.4f} | delta {full[m] - best[m]:+.4f}")
        clears = full["macro_f1"] > best["macro_f1"]
        print(f"#  ➜ {'YES' if clears else 'NO'} — the {k_max}-seed ensemble "
              f"{'BEATS' if clears else 'does NOT beat'} the best individual seed on macro-F1 "
              f"({full['macro_f1']:.4f} vs {best['macro_f1']:.4f}, {full['macro_f1'] - best['macro_f1']:+.4f}).")

        # --- 5. full ensemble gain over the MEAN seed vs the MEASURED std ---
        print("#")
        print(f"#  {k_max}-SEED ENSEMBLE gain over the MEAN individual seed, judged against the MEASURED seed-to-seed std:")
        for m in METRICS:
            g = full[m] - SEED_MEAN[m]
            sd = SEED_STD[m]
            print(f"#    {m:<18} gain {g:+.4f} vs std {sd:.4f} -> "
                  + ("larger than 1 std" if (len(done) > 1 and g > sd) else "NOT larger than 1 std"))
        g_f1, sd_f1 = full["macro_f1"] - SEED_MEAN["macro_f1"], SEED_STD["macro_f1"]
        genuine = len(done) > 1 and g_f1 > sd_f1
        print(f"#  ➜ macro-F1: {k_max}-seed ensemble {full['macro_f1']:.4f} vs mean seed {SEED_MEAN['macro_f1']:.4f} "
              f"(gain {g_f1:+.4f}); measured seed std {sd_f1:.4f} -> "
              + ("GENUINE improvement (gain > 1 measured seed std)." if genuine else
                 "NOT a genuine improvement by this standard (gain ≤ 1 measured seed std)."))
        print("#" * 100)
    else:
        print("\n⛔ no ensemble was computed (fewer than 2 seeds finished) — only the individual-seed statistics above.")
    if seed_failed:
        print(f"\n⚠️  failed seeds (excluded everywhere above): {[s for s, _ in seed_failed]}")
print(f"\n📄 full results CSV: {RESULTS_CSV} | per-seed probabilities: {PROBS_DIR}")


✅ CSV provenance: 5 seed row(s) from live test passes, 4 ensemble row(s) from saved probabilities | /kaggle/working/multi_seed_ensemble_results.csv
  INDIVIDUAL SEEDS (5 rows, training order)


,seed,accuracy,macro_f1,balanced_accuracy,macro_auroc,best_epoch,train_seconds
0,42,0.9080,0.9048,0.9022,0.9894,17,148.7
1,123,0.8876,0.8819,0.8793,0.9862,18,154.1
2,2024,0.8823,0.8783,0.8735,0.9866,12,174.8
3,7,0.9062,0.9021,0.9008,0.9896,14,156.4
4,99,0.8726,0.8675,0.8646,0.9840,20,164.6



  SEED-TO-SEED VARIANCE — mean ± sample std (ddof=1) across 5 independent seeds
  accuracy:           0.8913 +/- 0.0154 across 5 seeds (min 0.8726, max 0.9080, range 0.0354)
  macro_f1:           0.8869 +/- 0.0160 across 5 seeds (min 0.8675, max 0.9048, range 0.0373)
  balanced_accuracy:  0.8841 +/- 0.0168 across 5 seeds (min 0.8646, max 0.9022, range 0.0376)
  macro_auroc:        0.9872 +/- 0.0024 across 5 seeds (min 0.9840, max 0.9896, range 0.0057)
  -> the real seed-to-seed noise floor on macro-F1 is ±0.0160 (1 std), 32.0x the old ±0.0005 same-seed reproducibility placeholder.
  -> seed 42 (the known baseline) sits at macro-F1 0.9048, +1.12 std from the seed mean.

  ENSEMBLE SIZE CURVE — first k seeds (k = 1 is the first seed alone, normally seed 42)


,n_seeds,members,accuracy,macro_f1,balanced_accuracy,macro_auroc,Δ macro_f1 vs previous size,Δ macro_f1 vs mean seed,Δ macro_f1 vs best seed
0,1,42,0.9080,0.9048,0.9022,0.9894,NaN,0.0178,0.0000
1,2,42 123,0.9106,0.9069,0.9043,0.9907,0.0022,0.0200,0.0022
2,3,42 123 2024,0.9053,0.9018,0.8983,0.9914,-0.0051,0.0149,-0.0029
3,4,42 123 2024 7,0.9142,0.9109,0.9082,0.9922,0.0091,0.0240,0.0062
4,5,42 123 2024 7 99,0.9088,0.9056,0.9022,0.9919,-0.0053,0.0187,0.0009


  practical minimum: the smallest ensemble within 1 seed-std (0.0160) of the 5-seed ensemble's macro-F1 is k = 2 (macro-F1 0.9069 vs 0.9056).

####################################################################################################
#  5-SEED ENSEMBLE vs the BEST INDIVIDUAL SEED (seed 42, chosen by macro-F1)
####################################################################################################
#    accuracy           ensemble 0.9088 | best seed 0.9080 | delta +0.0009
#    macro_f1           ensemble 0.9056 | best seed 0.9048 | delta +0.0009
#    balanced_accuracy  ensemble 0.9022 | best seed 0.9022 | delta +0.0001
#    macro_auroc        ensemble 0.9919 | best seed 0.9894 | delta +0.0026
#  ➜ YES — the 5-seed ensemble BEATS the best individual seed on macro-F1 (0.9056 vs 0.9048, +0.0009).
#
#  5-SEED ENSEMBLE gain over the MEAN individual seed, judged against the MEASURED seed-to-seed std:
#    accuracy           gain +0.0175 vs std 0.0154 -> larger than 1 std
